# CS-781 Phase 3 (Rhetorical Roles) - SWEEP `p3s_gqa160`

**Phase-1 model:** dense GQA + RoPE decoder, 160M words, 1 epoch (111.1M params) - Phase-1 Codabench 0.8254

One Run All does, for every setup in `SWEEP` (cell 1):
1. **5-fold cross-validation** over the 90 labelled documents (45 CL + 45 IT; train and dev together). Each fold trains on 4/5 of the documents and predicts the held-out 1/5. One epoch is chosen per setup from the flat part of the mean held-out curve; scores (macro-F1 overall, CL only, IT only, precision, recall, per label) are computed from the held-out predictions of that epoch.
2. **Refit** of the best `REFIT_TOP_K` setups (plus `REFIT_ALWAYS`) on all 90 documents with the chosen number of epochs, prediction of the 10 test documents and a `submission_*.zip` (prediction.json at its root) for Codabench.

Every finished fold is cached in the bucket, so if the session stops, **Run All again resumes**. Nothing here uses test text for training.

**Kaggle:** GPU T4, Internet on, secrets `HF` and `HF_WRITE`. Set `QUICK_TEST = True` in cell 1 first for a few-minute check of the whole pipeline.
The sentence model: the Phase-1 network (vocabulary layer removed) reads token windows; each sentence is pooled to one vector; a BiLSTM over the sentence vectors of a unit of consecutive sentences, then a linear layer, gives the 13 role scores. Training recipe and all setting values are placeholders you may change (state changes in your report).

## 1. Settings

In [1]:
# [P3-PRELUDE:settings]
import json
# =====================================================================================================
# SETTINGS -- every value is a placeholder you can change. The training recipe (AdamW, two learning rates, warm-up + linear decay,
# class weights) is the Phase-2 recipe you approved and is NOT swept here.
# =====================================================================================================
SWEEP_NAME = "p3s_gqa160"
MODEL_KEY = "gqa160"
PHASE1_CHECKPOINT = "phase1_decoder_only_gqa_kvcache_full_layernorm_pre_adamw_rope_ctx2048_160Mw_a1bpe_160Mw_1ep_checkpoint.pt"
TOKENIZER_NAME = "a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json"
PEFT_TARGETS = ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.out_proj"]
IA3_OUT = ["attn.k_proj", "attn.v_proj"]      # key/value projections: their OUTPUT channels are rescaled
IA3_IN = ["ff.2"]        # feed-forward down-projection: its INPUT channels are rescaled

ATTENTION = "causal"   # only used by the Phase-1 decoder code below

BASE = dict(
    TECHNIQUE="lora", RANK=16, ALPHA=32, LORA_DROPOUT=0.0, PEFT_TARGETS=PEFT_TARGETS, LORMA_VARIANT="plus",
    ADALORA_INIT_RANK=24, ADALORA_TARGET_RANK=8, ADALORA_DELTA_T=10, ADALORA_ORTH=0.1,
    NF4_BLOCK=64, NF4_DOUBLE_QUANT=True, IA3_OUT=IA3_OUT, IA3_IN=IA3_IN, PROMPT_LEN=20, PREFIX_LEN=20,
    # context: the backbone reads WINDOW-token rows; a unit = SEQ_SENTS consecutive sentences of one document
    WINDOW=1024, PACK="together", SEQ_SENTS=24, TRAIN_STRIDE=24, INFER_STRIDE=12, MAX_SENT_TOKENS=256,
    MERGE="center",                       # sentences covered by two units: "center" = keep the unit where the sentence is most central
    POOL="mean+last", HEAD_TYPE="bilstm", HEAD_LSTM_HIDDEN=256, HEAD_DROPOUT=0.1, HEAD_NORM=True,
    DOMAIN_MODE="pooled",                 # "pooled" = one model on CL+IT;  "separate" = one CL model and one IT model (File B)
    # fixed training recipe
    BATCH=2, ACCUM=4, INFER_BATCH=4, EPOCHS=12, PATIENCE=0, LR=3e-4, HEAD_LR=1e-3, WEIGHT_DECAY=0.01, WARMUP_FRAC=0.06,
    GRAD_CLIP=1.0, AMP="fp16", SEED=0, CLASS_WEIGHTING="inv_sqrt", CW_CAP=10.0, LABEL_SMOOTHING=0.0,
    TRANS_LAMBDA=0.0, KD_ALPHA=0.0, KD_TEMPERATURE=2.0)

FULL = dict(LR=3e-5)    # full fine-tuning learning rate (approved: 3e-5)

SWEEP = [   # (name, settings that differ from BASE). Order = run order.
    ("lora_pooled", dict(TECHNIQUE='lora')),
    ("lora_separate", dict(TECHNIQUE='lora', DOMAIN_MODE='separate')),
    ("rslora", dict(TECHNIQUE='rslora')),
    ("lorma", dict(TECHNIQUE='lorma')),
    ("adalora", dict(TECHNIQUE='adalora')),
    ("qlora", dict(TECHNIQUE='qlora')),
    ("ia3", dict(TECHNIQUE='ia3')),
    ("prompt", dict(TECHNIQUE='prompt')),
    ("prefix", dict(TECHNIQUE='prefix')),
    ("full_ft", dict(TECHNIQUE='full_ft')),
    ("frozen_head_only", dict(TECHNIQUE='frozen')),
]

K_FOLDS = 5                   # document-level cross-validation folds over the 90 labelled documents (train + dev)
FOLD_SEED = 0
TRANS_LAMBDAS = [0.0, 0.3, 0.6, 1.0]   # decoding ablation on the saved held-out scores (0 = plain argmax; >0 = Viterbi with label transitions)
REFIT_TOP_K = 3               # refit the best K setups (by cross-validated macro-F1) on all 90 documents and write submission zips
REFIT_ALWAYS = ['lora_pooled', 'lora_separate']   # setup names that are always refitted (e.g. the two sides of the File A / File B check)
USE_ALL_GPUS = False          # True = split the cross-validation folds over all GPUs (Kaggle T4 x2: about twice as fast); False = one GPU
WORKERS_OVERRIDE = None       # testing only
MAX_SWEEP_HOURS = 9.0         # stop STARTING new folds after this many hours (finished folds are cached: Run All again resumes)
QUICK_TEST = False            # True = tiny end-to-end check (few short documents, 1 epoch, 256-token windows, first 2 setups)

if QUICK_TEST:
    SWEEP = SWEEP[:2]
    BASE.update(WINDOW=256, SEQ_SENTS=8, TRAIN_STRIDE=8, INFER_STRIDE=8, EPOCHS=1)
    SWEEP = [(n, dict(o, EPOCHS=1)) for n, o in SWEEP]
    K_FOLDS = 2
    REFIT_TOP_K = 1; REFIT_ALWAYS = []
    SWEEP_NAME += "_quicktest"
print(f"{SWEEP_NAME}: {len(SWEEP)} setups:", [n for n, _ in SWEEP])

p3s_gqa160: 11 setups: ['lora_pooled', 'lora_separate', 'rslora', 'lorma', 'adalora', 'qlora', 'ia3', 'prompt', 'prefix', 'full_ft', 'frozen_head_only']


## 2. Set-up: login, bucket

In [2]:
# [P3-PRELUDE:setup]
!pip install -q -U "huggingface_hub>=1.5"
import os, json, shutil, subprocess, sys, time, zipfile, random, math, gc
from collections import Counter, defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download, login, download_bucket_files, batch_bucket_files
from kaggle_secrets import UserSecretsClient

NB_START = time.time()
WORK = "/kaggle/working"
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_BUCKET = "kavincg23/cs781-models"
HF_CHECKPOINT_REPO = "kavincg23/cs781-checkpoints"   # old repo, read-only fallback
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
HF_WRITE_TOKEN = user_secrets.get_secret("HF_WRITE")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")


def get_dataset_file(filename):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename, repo_type="dataset", token=HF_TOKEN)


def fetch_model_file(name, local_path=None):
    local_path = local_path or f"{WORK}/{name}"
    if os.path.exists(local_path):
        return local_path
    part = local_path + ".part"
    try:
        download_bucket_files(HF_BUCKET, files=[(name, part)], raise_on_missing_files=True, token=HF_WRITE_TOKEN)
        os.replace(part, local_path)
        return local_path
    except Exception:
        if os.path.exists(part):
            os.remove(part)
    # fallback: the OLD model repo (the Phase-1 notebooks do the same; the tokenizer only lives there)
    p = hf_hub_download(repo_id=HF_CHECKPOINT_REPO, filename=name, repo_type="model", token=HF_WRITE_TOKEN)
    shutil.copy(p, local_path)
    print(f"(found {name} in the old repo {HF_CHECKPOINT_REPO})")
    return local_path


def upload_model_file(local_path, name=None):
    name = name or os.path.basename(local_path)
    try:
        batch_bucket_files(HF_BUCKET, add=[(local_path, name)], token=HF_WRITE_TOKEN)
        print(f"uploaded {name} to bucket {HF_BUCKET}")
    except Exception as e:
        print(f"(upload of {name} failed: {type(e).__name__}: {e}; the file is still in {WORK})")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 33.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.2.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.2.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.2.0 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.2.0 which is incompatible.
gradio-client 2.6.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.2.0 which is incompatible.
tokenizers 0.23.1 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.2.0 which is incompatible.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


device: cuda Tesla T4


## 4. Your Phase-1 BPE tokenizer (class unchanged), loaded from storage

In [3]:
# [P3-PRELUDE:bpe]
import array, heapq, time
from collections import Counter, defaultdict

BPE_MIN_WORD_FREQ = 1      # 1 = learn merges from every word type (exactly the Assignment-1 algorithm)
_END = "</w>"


class BPETokenizer:
    """Byte-level BPE (Assignment 1). Initial symbols are the UTF-8 bytes of each word, held as chr(b), plus
    '</w>' marking the end of a word so merges never cross word boundaries. Ties in pair frequency break
    lexicographically. Text is NOT lowercased."""

    def __init__(self, vocab_size):
        self.vocab_size = vocab_size
        self.merges = []
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.stoi = {}
        self._ranks = {}
        self._id_cache = {}

    # ---- helpers ----
    def _word_symbols(self, word):
        syms = [chr(b) for b in word.encode("utf-8")]
        syms.append(_END)
        return syms

    def _merge(self, symbols, pair):
        res, i = [], 0
        a, b = pair
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                res.append(a + b)
                i += 2
            else:
                res.append(symbols[i])
                i += 1
        return res

    # ---- training ----
    def fit_file(self, path, block_chars=50_000_000, min_word_freq=1):
        """Count word frequencies by streaming the corpus file in blocks, then learn the merges."""
        t0 = time.time()
        word_freqs = Counter()
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()          # the last word may be cut in half: finish it with the next block
                else:
                    leftover = ""
                word_freqs.update(parts)
        if leftover:
            word_freqs[leftover] += 1
        print(f"  counted {sum(word_freqs.values()):,} words ({len(word_freqs):,} distinct) in {time.time() - t0:.0f}s", flush=True)
        return self.fit_word_freqs(word_freqs, min_word_freq)

    def fit_word_freqs(self, word_freqs, min_word_freq=1):
        t0 = time.time()
        target = self.vocab_size
        words = [w for w, c in word_freqs.items() if c >= min_word_freq]
        freqs = [word_freqs[w] for w in words]
        splits = [self._word_symbols(w) for w in words]
        del word_freqs

        base = set()
        for syms in splits:
            base.update(syms)
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.merges = []
        self.itos.extend(sorted(base))
        print(f"  base byte alphabet: {len(base)} symbols; learning merges from {len(words):,} word types", flush=True)

        # pair_counts: how often each adjacent pair occurs (weighted by word frequency).
        # pair_words: which word types contain each pair, so a merge only revisits those words.
        pair_counts = {}
        pair_words = defaultdict(set)
        for i, syms in enumerate(splits):
            f = freqs[i]
            for j in range(len(syms) - 1):
                p = (syms[j], syms[j + 1])
                pair_counts[p] = pair_counts.get(p, 0) + f
                pair_words[p].add(i)

        # Max-heap of (-count, pair): pops the most frequent pair, ties -> lexicographically smallest pair
        # (identical to Assignment 1's min over (-count, pair)). Entries go stale when a count changes; a
        # fresh entry is pushed for every changed pair and stale ones are skipped when they reach the top.
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        while len(self.itos) < target:
            best, best_f = None, 0
            while heap:
                neg_c, p = heap[0]
                if pair_counts.get(p, 0) == -neg_c:
                    best, best_f = p, -neg_c
                    heapq.heappop(heap)
                    break
                heapq.heappop(heap)                  # stale entry
            if best is None:
                break
            if best_f < 2:
                print("  stopping: no pair occurs more than once", flush=True)
                break

            self.merges.append([best[0], best[1]])
            self.itos.append(best[0] + best[1])

            changed = set()
            for i in list(pair_words[best]):
                syms, f = splits[i], freqs[i]
                for j in range(len(syms) - 1):
                    p = (syms[j], syms[j + 1])
                    c = pair_counts.get(p, 0) - f
                    if c <= 0:
                        pair_counts.pop(p, None)
                    else:
                        pair_counts[p] = c
                    pair_words[p].discard(i)
                    changed.add(p)
                new_syms = self._merge(syms, best)
                splits[i] = new_syms
                for j in range(len(new_syms) - 1):
                    p = (new_syms[j], new_syms[j + 1])
                    pair_counts[p] = pair_counts.get(p, 0) + f
                    pair_words[p].add(i)
                    changed.add(p)
            for p in changed:
                c = pair_counts.get(p)
                if c:
                    heapq.heappush(heap, (-c, p))
            del pair_words[best]

            if len(self.merges) % 1000 == 0:
                print(f"  merges {len(self.merges):,} | vocab {len(self.itos):,} | {time.time() - t0:.0f}s", flush=True)

        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self._ranks = {(a, b): r for r, (a, b) in enumerate(self.merges)}
        self._id_cache = {}
        self.vocab_size = len(self.itos)      # the size the embedding table must actually have
        return self

    # ---- save / load (so the tokenizer is trained once and then reused) ----
    def save(self, path, meta=None):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"vocab_size": self.vocab_size, "merges": self.merges, "itos": self.itos,
                       "meta": meta or {}}, f)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(data["vocab_size"])
        tok.merges = [list(m) for m in data["merges"]]
        tok.itos = list(data["itos"])
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok._ranks = {(a, b): r for r, (a, b) in enumerate(tok.merges)}
        tok._id_cache = {}
        tok.vocab_size = len(tok.itos)
        return tok

    # ---- inference ----
    def _tokenize_word(self, word):
        syms = self._word_symbols(word)
        while len(syms) > 1:
            best_rank, best_j = None, -1
            for j in range(len(syms) - 1):
                r = self._ranks.get((syms[j], syms[j + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_j = r, j
            if best_rank is None:
                break
            syms = self._merge(syms, (syms[best_j], syms[best_j + 1]))
        return syms

    def _word_ids(self, word):
        ids = self._id_cache.get(word)
        if ids is None:
            unk = self.stoi["<unk>"]
            ids = [self.stoi.get(t, unk) for t in self._tokenize_word(word)]     # symbols never seen in training -> <unk>
            self._id_cache[word] = ids
        return ids

    def tokenize(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._tokenize_word(word))
        return out

    def get_vocab_size(self):
        return len(self.itos)

    def encode(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._word_ids(word))
        return out

    def encode_file(self, path, block_chars=5_000_000):
        """Encode a whole text file into a packed array of ids, block by block (bounded memory)."""
        out = array.array("i")
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()
                else:
                    leftover = ""
                cache = self._id_cache
                for word in parts:
                    ids = cache.get(word)
                    if ids is None:
                        ids = self._word_ids(word)
                    out.extend(ids)
        if leftover:
            out.extend(self._word_ids(leftover))
        return out

    def decode(self, ids):
        """ids -> string. Words are re-joined with single spaces; <unk> decodes to '?'."""
        words, cur = [], bytearray()
        for i in ids:
            i = int(i)
            if not 0 <= i < len(self.itos):
                continue
            tok = self.itos[i]
            if tok in ("<pad>", "<s>", "</s>"):
                continue
            if tok == "<unk>":
                cur.extend(b"?")
                continue
            ends_word = tok.endswith(_END)
            if ends_word:
                tok = tok[: -len(_END)]
            cur.extend(ord(c) & 0xFF for c in tok)
            if ends_word:
                words.append(bytes(cur).decode("utf-8", errors="replace"))
                cur = bytearray()
        if cur:
            words.append(bytes(cur).decode("utf-8", errors="replace"))
        return " ".join(words)



bpe_tokenizer = BPETokenizer.load(fetch_model_file(TOKENIZER_NAME))
VOCAB = bpe_tokenizer.get_vocab_size()
print("tokenizer:", TOKENIZER_NAME, "| vocab", VOCAB)

(…)full_v40000_160000000w_seed42_minf1.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

(found a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json in the old repo kavincg23/cs781-checkpoints)
tokenizer: a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json | vocab 40000


## 5. Phase-1 model code (copied unchanged from gqaRoPe160M1 (Phase-1 notebook))

In [4]:
# [P3-PRELUDE:model]
NORM_TYPE = "layernorm"; NORM_POSITION = "pre"   # as in the Phase-1 run
CONTEXT_LEN = 2048

class RMSNorm(nn.Module):
    """Like LayerNorm, but simpler: it only rescales by the root-mean-square
    of the values, it does not re-center the mean. Cheaper, and used in
    models like LLaMA."""

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight


def make_norm(dim, norm_type):
    """norm_type is "layernorm", "rmsnorm" or "deepnorm".
    deepnorm reuses a plain LayerNorm -- what makes deepnorm special isn't
    the norm formula, it's the residual scaling (alpha) and weight init
    (beta) we apply around it below."""
    if norm_type == "rmsnorm":
        return RMSNorm(dim)
    else:  # "layernorm" or "deepnorm"
        return nn.LayerNorm(dim)


def deepnorm_alpha(n_layers):
    """Residual scaling factor from the DeepNet paper: alpha = (2N)^0.25."""
    return (2.0 * n_layers) ** 0.25


def deepnorm_init(output_projections, n_layers):
    """Scales the init of a sub-layer's OUTPUT projection(s) (e.g. attention
    out_proj, the feed-forward's second linear) by beta = (8N)^-0.25, so a
    deep post-norm stack stays stable to train. Only used when
    NORM_TYPE == "deepnorm"."""
    beta = (8.0 * n_layers) ** -0.25
    for module in output_projections:
        nn.init.xavier_normal_(module.weight, gain=beta)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

# ---- Settings specific to this notebook ------------------------------
GQA_GROUP_SIZE = 4      # query heads that share one key/value head. 4 is exactly
                        # what ATTENTION_TYPE = "gqa" meant in the original notebooks.
CACHE_HEADROOM = 512    # slots reserved inside CONTEXT_LEN for GENERATED tokens.
                        # The prompt is cut to its last CONTEXT_LEN - CACHE_HEADROOM
                        # tokens; if generation fills the cache, the window is
                        # re-prefilled (see the markdown above for why). With
                        # CONTEXT_LEN = 2048 that leaves room for a 1536-token prompt
                        # (a 512-word legal prompt is roughly 700-1000 tokens) AND
                        # ~512 generated tokens, so in practice nothing is cut.
USE_NATIVE_GQA = False  # decided automatically by a micro-benchmark in the next
                        # section (True = pass enable_gqa=True to SDPA instead of
                        # expanding K/V with repeat_interleave)
POSITIONAL_ENCODING = "rope"   # "rope" | "learned"
                        # rope    = rotary embeddings: Q and K are rotated by a
                        #           position-dependent angle inside attention, so
                        #           attention scores depend on RELATIVE distance
                        #           (RoFormer, Su et al. 2021). No position table.
                        # learned = a trainable CONTEXT_LEN-row position table added to
                        #           the token embeddings (GPT-2 style, what the original
                        #           decoder-only notebook used).
ROPE_BASE = 10000.0     # RoPE frequency base: theta_i = ROPE_BASE ** (-2i / head_dim)
assert POSITIONAL_ENCODING in ("rope", "learned"), "POSITIONAL_ENCODING must be 'rope' or 'learned'"


def build_rope_cache(max_seq_len, head_dim, base=10000.0):
    """cos/sin tables for RoPE, each (max_seq_len, head_dim). Split-half
    convention: channel i is paired with channel i + head_dim/2, so the angle
    table is cat([freqs, freqs]) (NOT repeat_interleave, which pairs neighbours
    instead). Same construction as the RoPE written for Assignment 3."""
    assert head_dim % 2 == 0, "RoPE requires an even head dimension"
    inv_freq = 1.0 / (base ** (2 * torch.arange(0, head_dim // 2).float() / head_dim))
    positions = torch.arange(0, max_seq_len).float()
    freqs = torch.outer(positions, inv_freq)              # (max_seq_len, head_dim // 2)
    angles = torch.cat([freqs, freqs], dim=-1)            # (max_seq_len, head_dim)
    return angles.cos(), angles.sin()


def rotate_half(x):
    """Split the last dim in half and return (-x2, x1): a 90-degree rotation step."""
    x1, x2 = torch.chunk(x, 2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    """x_rotated = x * cos + rotate_half(x) * sin. x: (..., head_dim); cos/sin
    already broadcastable against x (the model gathers them per position). The
    arithmetic is done in fp32 (even under fp16 autocast) and cast back, so the
    rotation stays accurate and q/k keep the same dtype as v for SDPA."""
    xf = x.float()
    return (xf * cos + rotate_half(xf) * sin).to(x.dtype)


class KVCache:
    """Pre-allocated key/value storage for EVERY layer, written in place (the
    'static cache' idea from gpt-fast). Shapes are (batch, n_kv_heads, max_len,
    head_dim) -- n_kv_heads, not n_heads, which is where GQA's 4x saving
    lives. `length` is how many positions are currently filled (the same for
    every row in the batch: shorter prompts are left-padded)."""

    def __init__(self, n_layers, batch_size, n_kv_heads, max_len, head_dim, device, dtype):
        shape = (batch_size, n_kv_heads, max_len, head_dim)
        self.k = [torch.zeros(shape, device=device, dtype=dtype) for _ in range(n_layers)]
        self.v = [torch.zeros(shape, device=device, dtype=dtype) for _ in range(n_layers)]
        self.length = 0
        self.max_len = max_len

    def reset(self):
        self.length = 0

    def num_bytes(self):
        return sum(t.numel() * t.element_size() for t in self.k + self.v)


class CachedGQAttention(nn.Module):
    """Grouped-query self-attention (causal) with two entry points:
      forward()      -- the parallel form used for training and for PREFILL.
                        Optionally also stores this call's keys/values in a cache.
      decode_step()  -- ONE new token, reading/writing the cache.
    Parameter names (q_proj/k_proj/v_proj/out_proj) match the original
    MultiHeadAttention with ATTENTION_TYPE="gqa", so checkpoints are interchangeable."""

    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.group_size = n_heads // n_kv_heads

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.out_proj = nn.Linear(n_heads * self.head_dim, d_model)

    def forward(self, x, key_padding_mask=None, k_cache=None, v_cache=None, rope=None):
        """x: (batch, seq, d_model). key_padding_mask (optional): (batch, seq)
        bool, True = real (non-pad) position; leave None when nothing is
        padded so the fused kernels can run mask-free. If k_cache/v_cache are
        given, this call's (un-expanded) keys/values are written to
        positions [0:seq] of them -- that is the PREFILL step. rope (optional):
        (cos, sin), each (batch or 1, 1, seq, head_dim), gathered from the
        model's RoPE tables -- None when POSITIONAL_ENCODING == "learned"."""
        batch_size, seq_len, _ = x.shape
        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)

        if rope is not None:
            # rotate BEFORE caching: the cache stores already-rotated keys (each
            # key is rotated by its own position exactly once, when it is written)
            q = apply_rope(q, *rope)
            k = apply_rope(k, *rope)

        if k_cache is not None:
            k_cache[:, :, :seq_len] = k.to(k_cache.dtype)
            v_cache[:, :, :seq_len] = v.to(v_cache.dtype)

        attn_mask = None
        if key_padding_mask is not None:
            positions = torch.arange(seq_len, device=x.device)
            causal = positions.unsqueeze(1) >= positions.unsqueeze(0)               # (seq, seq)
            allowed = causal.unsqueeze(0) & key_padding_mask.unsqueeze(1)           # (batch, seq, seq)
            # a query may always see itself -- guarantees no row is fully
            # masked (which would produce NaNs for pad-position queries)
            allowed = allowed | torch.eye(seq_len, dtype=torch.bool, device=x.device).unsqueeze(0)
            attn_mask = allowed.unsqueeze(1)                                        # (batch, 1, seq, seq)

        use_causal_flag = attn_mask is None
        if USE_NATIVE_GQA and self.group_size > 1:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                                 is_causal=use_causal_flag, enable_gqa=True)
        else:
            if self.group_size > 1:
                k = k.repeat_interleave(self.group_size, dim=1)
                v = v.repeat_interleave(self.group_size, dim=1)
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                                 is_causal=use_causal_flag)

        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.n_heads * self.head_dim)
        return self.out_proj(out)

    def decode_step(self, x_t, k_cache, v_cache, cache_len, key_mask=None, rope=None):
        """x_t: (batch, d_model) -- ONE new position per row. Writes its K/V at
        index cache_len and attends over cache positions 0..cache_len
        (inclusive). key_mask (optional): (batch, cache_len + 1) bool, True =
        real key; None when the batch has no padding. rope (optional): (cos,
        sin), each (batch, head_dim) for this step's position."""
        batch_size = x_t.shape[0]

        # Fold the `group_size` query heads that share one K/V head into the
        # query-LENGTH axis: (batch, n_kv_heads, group_size, head_dim). The
        # cache is then used as-is (never expanded), and because the new token
        # is the last position no causal mask is needed.
        q = self.q_proj(x_t).view(batch_size, self.n_kv_heads, self.group_size, self.head_dim)
        k_new = self.k_proj(x_t).view(batch_size, self.n_kv_heads, self.head_dim)
        v_new = self.v_proj(x_t).view(batch_size, self.n_kv_heads, self.head_dim)

        if rope is not None:
            cos, sin = rope
            q = apply_rope(q, cos.view(batch_size, 1, 1, -1), sin.view(batch_size, 1, 1, -1))
            k_new = apply_rope(k_new, cos.view(batch_size, 1, -1), sin.view(batch_size, 1, -1))

        k_cache[:, :, cache_len] = k_new.to(k_cache.dtype)
        v_cache[:, :, cache_len] = v_new.to(v_cache.dtype)
        k = k_cache[:, :, :cache_len + 1]
        v = v_cache[:, :, :cache_len + 1]
        if q.dtype != k.dtype:
            q = q.to(k.dtype)

        attn_mask = None if key_mask is None else key_mask.view(batch_size, 1, 1, -1)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)   # (batch, n_kv, group, head_dim)
        out = out.reshape(batch_size, self.n_heads * self.head_dim).to(x_t.dtype)
        return self.out_proj(out)


class CachedDecoderBlock(nn.Module):
    """Same block as DecoderOnlyBlock (same norm / deepnorm wiring), with a
    decode_step() that mirrors forward() for one position."""

    def __init__(self, d_model, n_heads, d_ff, n_layers):
        super().__init__()
        self.norm_position = "post" if NORM_TYPE == "deepnorm" else NORM_POSITION

        n_kv_heads = max(1, n_heads // GQA_GROUP_SIZE)
        self.attn = CachedGQAttention(d_model, n_heads, n_kv_heads)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.norm1 = make_norm(d_model, NORM_TYPE)
        self.norm2 = make_norm(d_model, NORM_TYPE)

        if NORM_TYPE == "deepnorm":
            self.alpha = deepnorm_alpha(n_layers)
            deepnorm_init([self.attn.out_proj, self.ff[2]], n_layers)
        else:
            self.alpha = 1.0

    def forward(self, x, key_padding_mask=None, k_cache=None, v_cache=None, rope=None):
        if self.norm_position == "pre":
            x = x + self.attn(self.norm1(x), key_padding_mask=key_padding_mask,
                              k_cache=k_cache, v_cache=v_cache, rope=rope)
            x = x + self.ff(self.norm2(x))
        else:  # "post" (also deepnorm, with the alpha residual scale)
            x = self.norm1(x * self.alpha + self.attn(x, key_padding_mask=key_padding_mask,
                                                       k_cache=k_cache, v_cache=v_cache, rope=rope))
            x = self.norm2(x * self.alpha + self.ff(x))
        return x

    def decode_step(self, x_t, k_cache, v_cache, cache_len, key_mask=None, rope=None):
        """x_t: (batch, d_model). Norms and the FFN act per position, so only
        the attention part needs the cache."""
        if self.norm_position == "pre":
            x_t = x_t + self.attn.decode_step(self.norm1(x_t), k_cache, v_cache, cache_len, key_mask, rope)
            x_t = x_t + self.ff(self.norm2(x_t))
        else:
            x_t = self.norm1(x_t * self.alpha + self.attn.decode_step(x_t, k_cache, v_cache, cache_len, key_mask, rope))
            x_t = self.norm2(x_t * self.alpha + self.ff(x_t))
        return x_t


class CachedDecoderModel(nn.Module):
    """GPT-style decoder, GQA attention, KV-cache aware. Positions come from
    POSITIONAL_ENCODING: "learned" (pos_emb table added to the token
    embeddings) or "rope" (rotations inside attention, no table)."""

    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, context_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.use_rope = (POSITIONAL_ENCODING == "rope")
        if self.use_rope:
            cos, sin = build_rope_cache(context_len, d_model // n_heads, ROPE_BASE)
            # tables, not parameters; persistent=False keeps them out of state_dict
            # (one pair shared by every layer -- they depend only on position)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
        else:
            self.pos_emb = nn.Embedding(context_len, d_model)
        self.blocks = nn.ModuleList([
            CachedDecoderBlock(d_model, n_heads, d_ff, n_layers) for _ in range(n_layers)
        ])
        self.fc_out = nn.Linear(d_model, vocab_size)

    def _embed(self, token_ids, position_ids):
        """Token embedding (+ learned position embedding when not using RoPE)."""
        h = self.token_emb(token_ids)
        if not self.use_rope:
            h = h + self.pos_emb(position_ids)
        return h

    def forward(self, x, key_padding_mask=None, position_ids=None, last_position_only=False):
        """Training / full-recompute forward. x: (batch, seq) token ids.
        position_ids (optional, (batch, seq)) lets padded batches give each
        row's first real token position 0. last_position_only=True skips the
        big vocab projection for every position but the last (used by the
        uncached reference generator)."""
        batch_size, seq_len = x.shape
        if position_ids is None:
            position_ids = torch.arange(seq_len, device=x.device).unsqueeze(0)
        h = self._embed(x, position_ids)
        rope = None
        if self.use_rope:
            cos, sin = self.rope_cos[position_ids], self.rope_sin[position_ids]   # (batch or 1, seq, head_dim)
            rope = (cos.unsqueeze(1), sin.unsqueeze(1))                            # add the heads axis
        for block in self.blocks:
            h = block(h, key_padding_mask=key_padding_mask, rope=rope)
        if last_position_only:
            h = h[:, -1, :]
        return self.fc_out(h)

    def prefill(self, x, pad_lens, cache):
        """PREFILL. x: (batch, P) token ids, LEFT-padded; pad_lens: list of
        ints (pad count per row). Fills cache positions [0:P] for every layer,
        sets cache.length = P, and returns ONLY the last position's logits
        (batch, vocab) -- the one thing generation needs."""
        batch_size, prefill_len = x.shape
        assert prefill_len <= cache.max_len, "prompt window longer than the cache"
        pad_t = torch.tensor(pad_lens, dtype=torch.long, device=x.device)
        arange = torch.arange(prefill_len, device=x.device).unsqueeze(0)
        position_ids = (arange - pad_t.unsqueeze(1)).clamp(min=0)   # first REAL token -> position 0
        key_padding_mask = (arange >= pad_t.unsqueeze(1)) if any(p > 0 for p in pad_lens) else None

        h = self._embed(x, position_ids)
        rope = None
        if self.use_rope:
            cos, sin = self.rope_cos[position_ids], self.rope_sin[position_ids]
            rope = (cos.unsqueeze(1), sin.unsqueeze(1))
        for layer_idx, block in enumerate(self.blocks):
            h = block(h, key_padding_mask=key_padding_mask,
                      k_cache=cache.k[layer_idx], v_cache=cache.v[layer_idx], rope=rope)
        cache.length = prefill_len
        return self.fc_out(h[:, -1, :])

    def decode_step(self, token_ids, pad_t, has_padding, cache):
        """DECODE. token_ids: (batch,) -- ONE new token per row. pad_t: (batch,)
        long tensor of pad counts. Appends to the cache, returns logits
        (batch, vocab) for the NEXT token. O(1) new work per step (plus one
        read of the cache)."""
        cache_len = cache.length
        assert cache_len < cache.max_len, "cache is full -- re-prefill before decoding further"
        position_ids = (cache_len - pad_t).clamp(min=0)              # (batch,)
        key_mask = None
        if has_padding:
            key_mask = torch.arange(cache_len + 1, device=token_ids.device).unsqueeze(0) >= pad_t.unsqueeze(1)

        h = self._embed(token_ids, position_ids)
        rope = (self.rope_cos[position_ids], self.rope_sin[position_ids]) if self.use_rope else None   # each (batch, head_dim)
        for layer_idx, block in enumerate(self.blocks):
            h = block.decode_step(h, cache.k[layer_idx], cache.v[layer_idx], cache_len, key_mask, rope)
        cache.length = cache_len + 1
        return self.fc_out(h)

## 6. Backbone wrapper (the Phase-2 wrapper: runs the Phase-1 blocks and returns hidden states instead of next-token scores)

In [5]:
# [P3-PRELUDE:wrapper]
# ===================== Backbone wrapper for the Phase-1 decoder models (GQA / MoE CachedDecoderModel) =====================
# The Phase-1 model code above is UNCHANGED. For NER we only (a) run its blocks and take the hidden states instead of the
# next-token logits, and (b) optionally replace the attention's forward with one that can drop the causal mask
# ("bidirectional") and/or prepend learned prefix keys/values (prefix tuning). With both options off it calls the
# original forward unchanged.

_ORIG_GQA_FORWARD = CachedGQAttention.forward


def ner_gqa_forward(self, x, key_padding_mask=None, k_cache=None, v_cache=None, rope=None):
    bidir = getattr(self, "ner_bidirectional", False)
    pk = getattr(self, "prefix_k", None)
    if not bidir and pk is None:
        return _ORIG_GQA_FORWARD(self, x, key_padding_mask=key_padding_mask, k_cache=k_cache, v_cache=v_cache, rope=rope)
    B, T, _ = x.shape
    q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
    v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
    if rope is not None:
        q = apply_rope(q, *rope)
        k = apply_rope(k, *rope)
    P = 0
    if pk is not None:                                          # learned prefix keys/values (no position: never rotated)
        P = pk.shape[1]
        k = torch.cat([pk.unsqueeze(0).expand(B, -1, -1, -1).to(k.dtype), k], dim=2)
        v = torch.cat([self.prefix_v.unsqueeze(0).expand(B, -1, -1, -1).to(v.dtype), v], dim=2)
    if self.group_size > 1:
        k = k.repeat_interleave(self.group_size, dim=1)
        v = v.repeat_interleave(self.group_size, dim=1)
    if bidir and P == 0 and key_padding_mask is None:          # full-attention fast path (no mask tensor at all)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_heads * self.head_dim)
        return self.out_proj(out)
    pos = torch.arange(T, device=x.device)
    allowed = torch.ones(T, T, dtype=torch.bool, device=x.device) if bidir else (pos.unsqueeze(1) >= pos.unsqueeze(0))
    allowed = allowed.unsqueeze(0).expand(B, T, T)
    if key_padding_mask is not None:
        allowed = allowed & key_padding_mask.unsqueeze(1)
        allowed = allowed | torch.eye(T, dtype=torch.bool, device=x.device).unsqueeze(0)
    if P:
        allowed = torch.cat([torch.ones(B, T, P, dtype=torch.bool, device=x.device), allowed], dim=2)
    out = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed.unsqueeze(1))
    out = out.transpose(1, 2).contiguous().view(B, T, self.n_heads * self.head_dim)
    return self.out_proj(out)


CachedGQAttention.forward = ner_gqa_forward


class DecoderBackbone(nn.Module):
    def __init__(self, lm, bidirectional=False):
        super().__init__()
        self.original_param_count = sum(p.numel() for p in lm.parameters())
        self.lm = lm
        self.causal = not bidirectional
        for blk in lm.blocks:
            blk.attn.ner_bidirectional = bidirectional
        if isinstance(lm.fc_out, nn.Sequential):                # the final LayerNorm is kept; the vocab projection is dropped
            self.final_norm = lm.fc_out[0]
        else:
            self.final_norm = nn.Identity()
        lm.fc_out = nn.Identity()
        self.d_model = lm.token_emb.embedding_dim
        self.max_len = lm.rope_cos.shape[0] if lm.use_rope else lm.pos_emb.num_embeddings

    def layer_list(self):
        return list(self.lm.blocks)

    def final_modules_params(self):
        return list(self.final_norm.parameters())

    def embedding_weight(self):
        return self.lm.token_emb.weight

    def adapter_sites(self, kind):
        d = self.d_model
        if kind == "houlsby":                                   # after attention AND after the feed-forward, every block
            return [(m, d) for blk in self.lm.blocks for m in (blk.attn, blk.ff)]
        if kind == "pfeiffer":                                  # after the feed-forward only
            return [(blk.ff, d) for blk in self.lm.blocks]
        raise ValueError(kind)

    def enable_prefix(self, P, hidden=0):
        for blk in self.lm.blocks:
            a = blk.attn
            a.prefix_k = nn.Parameter(torch.randn(a.n_kv_heads, P, a.head_dim) * 0.02)
            a.prefix_v = nn.Parameter(torch.randn(a.n_kv_heads, P, a.head_dim) * 0.02)

    def features(self, ids, mask, soft_prompt=None):
        lm = self.lm
        h = lm.token_emb(ids)
        B = ids.shape[0]
        P = 0
        if soft_prompt is not None:
            P = soft_prompt.shape[0]
            h = torch.cat([soft_prompt.unsqueeze(0).expand(B, -1, -1).to(h.dtype), h], dim=1)
            mask = torch.cat([torch.ones(B, P, dtype=torch.bool, device=mask.device), mask], dim=1)
        T = h.shape[1]
        assert T <= self.max_len, f"window ({T} incl. {P} prompt tokens) longer than the model's {self.max_len} positions"
        pos = torch.arange(T, device=ids.device)
        rope = None
        if lm.use_rope:
            rope = (lm.rope_cos[pos].unsqueeze(0).unsqueeze(0), lm.rope_sin[pos].unsqueeze(0).unsqueeze(0))
        else:
            h = h + lm.pos_emb(pos).unsqueeze(0)
        # right padding: with causal attention real tokens never see the pads, so no mask is needed (fast fused kernel);
        # bidirectional attention needs the padding mask
        kpm = None if (self.causal or bool(mask.all())) else mask
        for blk in lm.blocks:
            h = blk(h, key_padding_mask=kpm, rope=rope)
        h = self.final_norm(h)
        return h[:, P:]


@torch.no_grad()
def lm_sanity_loss(lm, docs, device, n_windows=6, length=1024):
    """Next-token loss of the loaded Phase-1 model on a few NER-document windows (BEFORE the LM head is dropped).
    A correctly loaded model gives roughly its Phase-1 held-out loss (~2.5-3.5 nats); random weights give ~10.6."""
    lm.eval().to(device)
    tot, cnt = 0.0, 0
    for d in list(docs.values())[:n_windows]:
        ids = torch.tensor(d["ids"][:length + 1], dtype=torch.long, device=device).unsqueeze(0)
        if ids.shape[1] < 2:
            continue
        logits = lm(ids[:, :-1]).float()
        tot += F.cross_entropy(logits.view(-1, logits.shape[-1]), ids[:, 1:].reshape(-1), reduction="sum").item()
        cnt += ids.shape[1] - 1
    return tot / max(cnt, 1)


def build_phase1_lm():
    """The Phase-1 dense GQA model exactly as trained (111.1M params, no final LayerNorm), with its trained weights."""
    torch.manual_seed(0)
    lm = CachedDecoderModel(VOCAB, d_model=768, n_heads=12, n_layers=8, d_ff=3072, context_len=CONTEXT_LEN)
    lm.load_state_dict(PHASE1_STATE, strict=True)
    return lm


def make_backbone(lm):
    return DecoderBackbone(lm, bidirectional=(ATTENTION == "bidirectional"))

## 7. Phase-3 engine: sentence units, pooling, BiLSTM head, training, decoding, scoring, folds, submission writer

In [6]:
# [P3-PRELUDE:core]
# ===================== Phase 3 (RR) engine: sentence units, pooling, heads, folds, training, inference, scoring =====================
# Works with any Phase-1 model wrapped as a "backbone" exposing  features(ids, mask, soft_prompt) -> (B, T, d_model).
# Data model: a document = ordered sentences; a "unit" = up to SEQ_SENTS consecutive sentences of one document. The backbone reads the
# sentences of a unit either one by one (PACK="alone") or packed together into windows of <= WINDOW tokens (PACK="together");
# every sentence is pooled into one vector; an optional BiLSTM runs over the sentence vectors of the unit; a linear layer gives 13 scores.
import math, time, random, json, os, copy, zipfile
from collections import Counter, defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F

IGNORE = -100


# ----------------------------------------------------------------------------------------------------------------------
# 1. Labels and documents
# ----------------------------------------------------------------------------------------------------------------------
def labels_from_vocab(vocab_raw):
    """label_vocab.json is a dict label -> int (Fact 0 ... None 12). Returns the label list ordered by that int."""
    if isinstance(vocab_raw, dict):
        vals = list(vocab_raw.values())
        if all(isinstance(v, int) for v in vals):
            labels = [k for k, _ in sorted(vocab_raw.items(), key=lambda kv: kv[1])]
        else:
            labels = list(vocab_raw.keys())
    else:
        labels = list(vocab_raw)
    assert all(isinstance(l, str) for l in labels), "a label is not a string (the 'None' problem?)"
    assert "None" in labels and len(set(labels)) == len(labels)
    return labels


def as_doc_list(obj, domain):
    """Train/dev files: list of {id, text, labels}. Test files: dict {id: [sentences]}. Returns [{id, domain, text, labels|None}]."""
    out = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            out.append(dict(id=k, domain=domain, text=list(v), labels=None))
    else:
        for r in obj:
            out.append(dict(id=r["id"], domain=domain, text=list(r["text"]), labels=list(r["labels"]) if r.get("labels") is not None else None))
    return out


def encode_docs(tok, docs, label2id, max_sent_tokens):
    """Tokenise every sentence with the Phase-1 BPE (sentences longer than max_sent_tokens keep their first tokens)."""
    out, n_trunc, n_sent = [], 0, 0
    for d in docs:
        sents = []
        for s in d["text"]:
            ids = tok.encode(s)
            if not ids:
                ids = [1]                                        # <unk>: a sentence must have at least one token
            if len(ids) > max_sent_tokens:
                ids = ids[:max_sent_tokens]; n_trunc += 1
            sents.append(ids); n_sent += 1
        y = None if d.get("labels") is None else [label2id[l] for l in d["labels"]]
        if y is not None:
            assert len(y) == len(sents), f"{d['id']}: {len(y)} labels for {len(sents)} sentences"
        out.append(dict(id=d["id"], domain=d["domain"], sents=sents, y=y, n=len(sents)))
    return out, dict(sentences=n_sent, truncated=n_trunc)


# ----------------------------------------------------------------------------------------------------------------------
# 2. Units, packing, collation
# ----------------------------------------------------------------------------------------------------------------------
def make_units(n, seq_sents, stride):
    """Ranges (s, e) of at most seq_sents consecutive sentences that cover 0..n-1; consecutive ranges start `stride` apart; the last
    range is aligned to the end of the document."""
    if n <= 0:
        return []
    if n <= seq_sents:
        return [(0, n)]
    starts = list(range(0, n - seq_sents + 1, max(1, stride)))
    if starts[-1] + seq_sents < n:
        starts.append(n - seq_sents)
    return [(s, s + seq_sents) for s in starts]


def pack_groups(lens, window, together):
    """Indices of the sentences of a unit, grouped into backbone rows of at most `window` tokens (a sentence is never split)."""
    if not together:
        return [[i] for i in range(len(lens))]
    groups, cur, tot = [], [], 0
    for i, l in enumerate(lens):
        if cur and tot + l > window:
            groups.append(cur); cur, tot = [], 0
        cur.append(i); tot += l
    if cur:
        groups.append(cur)
    return groups


def collate(docs, units, C, with_labels=True, pad_id=0):
    """units: list of (doc_index, s, e). Builds the tensors for one batch:
       ids/mask (R, T): backbone rows (right padded); gsid (R, T): global sentence index of every token (-1 = pad);
       last_rows/last_pos (S,): position of every sentence's last token; unit_sizes (U,), unit_mask (U, Lmax),
       labels (U, Lmax) (IGNORE for padding), kd (U, Lmax, K) if the documents carry teacher scores."""
    window, together = C["WINDOW"], C["PACK"] == "together"
    rows_ids, rows_sid, last_rows, last_pos, sizes = [], [], [], [], []
    g0 = 0
    for di, s, e in units:
        sents = docs[di]["sents"][s:e]
        for grp in pack_groups([len(x) for x in sents], window, together):
            ids, sid = [], []
            for i in grp:
                ids += sents[i]; sid += [g0 + i] * len(sents[i])
                last_rows.append(len(rows_ids)); last_pos.append(len(ids) - 1)
            rows_ids.append(ids); rows_sid.append(sid)
        sizes.append(e - s); g0 += e - s
    R, T = len(rows_ids), max(len(r) for r in rows_ids)
    ids = torch.full((R, T), pad_id, dtype=torch.long)
    mask = torch.zeros((R, T), dtype=torch.bool)
    gsid = torch.full((R, T), -1, dtype=torch.long)
    for r, (a, b) in enumerate(zip(rows_ids, rows_sid)):
        ids[r, :len(a)] = torch.tensor(a); mask[r, :len(a)] = True; gsid[r, :len(b)] = torch.tensor(b)
    U, L = len(units), max(sizes)
    unit_mask = torch.arange(L).unsqueeze(0) < torch.tensor(sizes).unsqueeze(1)
    b = dict(ids=ids, mask=mask, gsid=gsid, last_rows=torch.tensor(last_rows), last_pos=torch.tensor(last_pos),
             n_sents=g0, unit_sizes=sizes, unit_mask=unit_mask)
    if with_labels and all(docs[di]["y"] is not None for di, _, _ in units):
        lab = torch.full((U, L), IGNORE, dtype=torch.long)
        for u, (di, s, e) in enumerate(units):
            lab[u, :e - s] = torch.tensor(docs[di]["y"][s:e])
        b["labels"] = lab
    if with_labels and all(docs[di].get("kd") is not None for di, _, _ in units):
        K = docs[units[0][0]]["kd"].shape[-1]
        kd = torch.zeros((U, L, K))
        for u, (di, s, e) in enumerate(units):
            kd[u, :e - s] = torch.as_tensor(docs[di]["kd"][s:e]).float()
        b["kd"] = kd
    return b


def to_device(b, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}


# ----------------------------------------------------------------------------------------------------------------------
# 3. Model = backbone + sentence pooling + head
# ----------------------------------------------------------------------------------------------------------------------
def pool_sentences(h, b, mode):
    """h: (R, T, d) backbone states -> (S, d_pool): mean of the sentence's tokens and/or its last token (a causal model has read the
    whole sentence at its last token)."""
    S, parts = b["n_sents"], []
    if mode in ("mean", "mean+last"):
        valid = b["gsid"] >= 0
        hv, sid = h[valid].float(), b["gsid"][valid]
        sums = torch.zeros(S, hv.shape[1], device=h.device).index_add_(0, sid, hv)
        cnt = torch.zeros(S, device=h.device).index_add_(0, sid, torch.ones(sid.shape[0], device=h.device))
        parts.append(sums / cnt.clamp(min=1).unsqueeze(1))
    if mode in ("last", "mean+last"):
        parts.append(h[b["last_rows"], b["last_pos"]].float())
    assert parts, f"unknown POOL {mode}"
    return torch.cat(parts, -1)


class BiLSTMLayer(nn.Module):
    """Bidirectional LSTM over the sentence vectors of a unit (padding packed away, so the backward direction never reads pads)."""

    def __init__(self, d_in, hidden, layers=1, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(d_in, hidden, num_layers=layers, batch_first=True, bidirectional=True, dropout=dropout if layers > 1 else 0.0)
        self.out_dim = 2 * hidden

    def forward(self, x, mask):
        lengths = mask.sum(1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(x.float(), lengths, batch_first=True, enforce_sorted=False)
        with torch.autocast(device_type=x.device.type, enabled=False):
            out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.shape[1])
        return out


class SentenceHead(nn.Module):
    def __init__(self, d_in, n_labels, head_type="bilstm", lstm_hidden=256, lstm_layers=1, dropout=0.1, norm=True):
        super().__init__()
        self.norm = nn.LayerNorm(d_in) if norm else nn.Identity()
        self.drop = nn.Dropout(dropout)
        self.lstm = None
        d_out = d_in
        if head_type == "bilstm":
            self.lstm = BiLSTMLayer(d_in, lstm_hidden, lstm_layers, dropout)
            d_out = self.lstm.out_dim
        elif head_type != "linear":
            raise ValueError(head_type)
        self.out = nn.Linear(d_out, n_labels)

    def forward(self, v, mask):
        x = self.drop(self.norm(v))
        if self.lstm is not None:
            x = self.lstm(x, mask)
        return self.out(x)


class RRModel(nn.Module):
    """Same attribute names as the Phase-2 NERModel (backbone, soft_prompt, head_parameters) so the technique code can treat both alike."""

    def __init__(self, backbone, d_model, n_labels, C):
        super().__init__()
        self.backbone = backbone
        self.soft_prompt = None
        self.pool = C["POOL"]
        d_pool = d_model * (2 if self.pool == "mean+last" else 1)
        self.sent_head = SentenceHead(d_pool, n_labels, C["HEAD_TYPE"], C["HEAD_LSTM_HIDDEN"], 1, C["HEAD_DROPOUT"], C["HEAD_NORM"])
        self.n_labels = n_labels

    def head_parameters(self):
        return list(self.sent_head.parameters())

    def forward(self, b):
        h = self.backbone.features(b["ids"], b["mask"], self.soft_prompt)
        v = pool_sentences(h, b, self.pool)                                  # (S, d_pool), units in order
        um = b["unit_mask"]
        vp = v.new_zeros(um.shape[0], um.shape[1], v.shape[1])
        vp[um] = v                                                           # row-major order of True entries == global sentence order
        return self.sent_head(vp, um)                                        # (U, Lmax, n_labels)


# ----------------------------------------------------------------------------------------------------------------------
# 4. Loss, class weights, training
# ----------------------------------------------------------------------------------------------------------------------
def class_weights(train_docs, n_labels, scheme, cap):
    if scheme == "none":
        return None
    cnt = torch.zeros(n_labels)
    for d in train_docs:
        for y in d["y"]:
            cnt[y] += 1
    present = cnt > 0
    n = cnt.sum()
    if scheme == "inv_sqrt":
        w = torch.sqrt(n / (present.sum() * cnt.clamp(min=1)))
    elif scheme == "inv":
        w = n / (present.sum() * cnt.clamp(min=1))
    else:
        raise ValueError(scheme)
    w = torch.where(present, w.clamp(max=cap), torch.zeros_like(w))
    return w / w[present].mean()                                           # mean weight over the labels present = 1


def sentence_loss(logits, labels, cw, C):
    K = logits.shape[-1]
    return F.cross_entropy(logits.reshape(-1, K).float(), labels.reshape(-1), weight=cw, ignore_index=IGNORE,
                           label_smoothing=C.get("LABEL_SMOOTHING", 0.0))


def _lr_lambda(total, warmup):
    def f(step):
        if step < warmup:
            return (step + 1) / max(1, warmup)
        return max(0.0, (total - step) / max(1, total - warmup))
    return f


def trainable_state(model):
    """Everything that can differ from the Phase-1 weights: trainable parameters + routing buffers that fine-tuning may update."""
    st = {n: p.detach().to("cpu", copy=True) for n, p in model.named_parameters() if p.requires_grad}
    for n, bf in model.named_buffers():
        if n.endswith(("select_bias", ".tau", ".tau_ready", ".ada_mask")):
            st[n] = bf.detach().to("cpu", copy=True)
    return st


def load_trainable_state(model, st):
    known = dict(model.named_parameters()); known.update(dict(model.named_buffers()))
    missing = [k for k in st if k not in known]
    assert not missing, f"state has unknown keys: {missing[:5]}"
    model.load_state_dict(st, strict=False)


def train_rr(model, train_docs, C, device, eval_fn=None, epochs=None, log=print, keep_best=True):
    """Trains the trainable parameters on the units of train_docs. eval_fn(model) -> (score, details) is called after every epoch; the
    best epoch's state is kept. Optional model hooks (used by AdaLoRA): extra_loss(), on_before_step(step, total)."""
    epochs = epochs or C["EPOCHS"]
    model.to(device)
    torch.manual_seed(C["SEED"]); random.seed(C["SEED"])
    items = [(di, s, e) for di, d in enumerate(train_docs) for (s, e) in make_units(d["n"], C["SEQ_SENTS"], C["TRAIN_STRIDE"])]
    cw = class_weights(train_docs, model.n_labels, C["CLASS_WEIGHTING"], C["CW_CAP"])
    cw = cw.to(device) if cw is not None else None
    head_params = [p for p in model.head_parameters() if p.requires_grad]
    hp = set(id(p) for p in head_params)
    other = [p for p in model.parameters() if p.requires_grad and id(p) not in hp]
    groups = [dict(params=head_params, lr=C["HEAD_LR"])]
    if other:
        groups.append(dict(params=other, lr=C["LR"]))
    opt = torch.optim.AdamW(groups, weight_decay=C["WEIGHT_DECAY"], betas=(0.9, 0.98))
    steps_per_epoch = math.ceil(len(items) / (C["BATCH"] * C["ACCUM"]))
    total = steps_per_epoch * epochs
    sched = torch.optim.lr_scheduler.LambdaLR(opt, _lr_lambda(total, int(C["WARMUP_FRAC"] * total)))
    use_amp = device.type == "cuda" and C["AMP"] == "fp16"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    hist, best, best_state, best_ep, bad = [], -1.0, None, 0, 0
    t0, tokens, gstep = time.time(), 0, 0
    kd_on = C.get("KD_ALPHA", 0) > 0 and all(d.get("kd") is not None for d in train_docs)
    log(f"training: {len(items)} units/epoch, {steps_per_epoch} optimizer steps/epoch, {epochs} epochs" + (", KD on" if kd_on else ""))
    for ep in range(1, epochs + 1):
        model.train()
        order = items[:]; random.Random(C["SEED"] * 1000 + ep).shuffle(order)
        run_loss, nb, skipped, pending = 0.0, 0, 0, 0
        opt.zero_grad(set_to_none=True)
        for bi in range(0, len(order), C["BATCH"]):
            b = to_device(collate(train_docs, order[bi:bi + C["BATCH"]], C), device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(b)
            logits = logits.float()
            loss = sentence_loss(logits, b["labels"], cw, C)
            if kd_on:
                T = C["KD_TEMPERATURE"]; um = b["unit_mask"]
                kd = F.kl_div(F.log_softmax(logits[um] / T, -1), F.softmax(b["kd"][um] / T, -1), reduction="batchmean") * T * T
                loss = (1 - C["KD_ALPHA"]) * loss + C["KD_ALPHA"] * kd
            if hasattr(model, "extra_loss"):
                loss = loss + model.extra_loss()
            if not torch.isfinite(loss):
                skipped += 1
                continue
            scaler.scale(loss / C["ACCUM"]).backward()
            nb += 1; pending += 1; tokens += int(b["mask"].sum()); run_loss += loss.item()
            if pending >= C["ACCUM"] or (bi + C["BATCH"] >= len(order) and pending > 0):
                scaler.unscale_(opt)
                if hasattr(model, "on_before_step"):
                    model.on_before_step(gstep, total)
                torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], C["GRAD_CLIP"])
                scaler.step(opt); scaler.update(); sched.step(); gstep += 1
                opt.zero_grad(set_to_none=True); pending = 0
        if pending > 0:                                   # gradients left over (the last batch was non-finite): use them, never drop them
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], C["GRAD_CLIP"])
            scaler.step(opt); scaler.update(); sched.step(); gstep += 1
            opt.zero_grad(set_to_none=True); pending = 0
        row = dict(epoch=ep, train_loss=round(run_loss / max(1, nb), 5), seconds=round(time.time() - t0, 1))
        if skipped:
            row["skipped_nonfinite_batches"] = skipped
        if eval_fn is not None:
            score, det = eval_fn(model)
            row.update(heldout_macroF1=round(score, 5), **det)
            if score > best:
                best, best_ep, best_state, bad = score, ep, (trainable_state(model) if keep_best else None), 0
            else:
                bad += 1
        hist.append(row)
        log("  epoch %d: " % ep + ", ".join(f"{k}={v}" for k, v in row.items() if k != "epoch"))
        if nb == 0:
            log("  every batch of this epoch was non-finite: stopping"); break
        if eval_fn is not None and C["PATIENCE"] and bad >= C["PATIENCE"]:
            log(f"  early stop: no improvement for {bad} epochs"); break
    if eval_fn is None:
        best_state, best_ep = trainable_state(model), len(hist)
    secs = time.time() - t0
    peak = torch.cuda.max_memory_allocated(device) / 1e9 if device.type == "cuda" else None
    return dict(history=hist, best_epoch=best_ep, best_score=best, best_state=best_state, train_seconds=round(secs, 1),
                peak_mem_gb=round(peak, 3) if peak else None, tokens_per_sec=round(tokens / max(secs, 1e-9), 1))


# ----------------------------------------------------------------------------------------------------------------------
# 5. Inference and decoding
# ----------------------------------------------------------------------------------------------------------------------
@torch.no_grad()
def predict_logprobs(model, docs, C, device):
    """Per document a (n_sentences, K) tensor of log-probabilities. Sentences covered by several overlapping units are merged:
    MERGE='center' keeps the unit in which the sentence is farthest from the unit's edges, 'average' averages the log-probabilities."""
    model.eval(); model.to(device)
    K = model.n_labels
    units = [(di, s, e) for di, d in enumerate(docs) for (s, e) in make_units(d["n"], C["SEQ_SENTS"], C["INFER_STRIDE"])]
    units.sort(key=lambda u: sum(len(x) for x in docs[u[0]]["sents"][u[1]:u[2]]))      # similar lengths together: less padding
    acc = [torch.zeros(d["n"], K) for d in docs]
    best = [torch.full((d["n"],), -1.0) for d in docs]
    cnt = [torch.zeros(d["n"]) for d in docs]
    use_amp = device.type == "cuda" and C["AMP"] == "fp16"
    for bi in range(0, len(units), C["INFER_BATCH"]):
        chunk = units[bi:bi + C["INFER_BATCH"]]
        b = to_device(collate(docs, chunk, C, with_labels=False), device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            lg = model(b)
        lp = F.log_softmax(lg.float(), -1).cpu()
        for u, (di, s, e) in enumerate(chunk):
            L = e - s
            pos = torch.arange(L)
            if C["MERGE"] == "average":
                acc[di][s:e] += lp[u, :L]; cnt[di][s:e] += 1
            else:
                score = torch.minimum(pos, L - 1 - pos).float()
                upd = score > best[di][s:e]
                acc[di][s:e][upd] = lp[u, :L][upd]
                best[di][s:e] = torch.where(upd, score, best[di][s:e])
    if C["MERGE"] == "average":
        acc = [a / c.clamp(min=1).unsqueeze(1) for a, c in zip(acc, cnt)]
    return acc


def learn_logT(train_docs, K, smoothing=1.0):
    """Label-transition log-probabilities log P(y_t | y_{t-1}) from the training documents (add-`smoothing`)."""
    cnt = torch.full((K, K), float(smoothing))
    for d in train_docs:
        for a, b in zip(d["y"][:-1], d["y"][1:]):
            cnt[a, b] += 1
    return (cnt / cnt.sum(1, keepdim=True)).log()


def decode_ids(lp, logT=None, lam=0.0):
    """Label sequence for one document: per-sentence argmax, or Viterbi over lp + lam * logT (smooths isolated label flips)."""
    n = lp.shape[0]
    if n == 0:
        return []
    if logT is None or lam == 0.0:
        return lp.argmax(-1).tolist()
    K = lp.shape[1]
    T = lam * logT
    score, back = lp[0].clone(), torch.zeros(n, K, dtype=torch.long)
    for t in range(1, n):
        s, arg = (score.unsqueeze(1) + T).max(0)
        back[t] = arg; score = s + lp[t]
    path = [int(score.argmax())]
    for t in range(n - 1, 0, -1):
        path.append(int(back[t, path[-1]]))
    return path[::-1]


# ----------------------------------------------------------------------------------------------------------------------
# 6. Scoring (macro-F1 overall and per domain)
# ----------------------------------------------------------------------------------------------------------------------
def macro_scores(gold, pred, labels):
    """gold/pred: flat lists of label ids. macro_F1 averages over the labels that occur in gold OR pred (scikit-learn's default);
    macro_F1_gold over the labels in gold; macro_F1_all13 over all labels. The official aggregation is not known until a submission."""
    K = len(labels)
    tp, fp, fn = [0] * K, [0] * K, [0] * K
    for g, p in zip(gold, pred):
        if g == p:
            tp[g] += 1
        else:
            fp[p] += 1; fn[g] += 1
    f1, P, R, per = [0.0] * K, [0.0] * K, [0.0] * K, {}
    for c in range(K):
        P[c] = tp[c] / (tp[c] + fp[c]) if tp[c] + fp[c] else 0.0
        R[c] = tp[c] / (tp[c] + fn[c]) if tp[c] + fn[c] else 0.0
        f1[c] = 2 * tp[c] / (2 * tp[c] + fp[c] + fn[c]) if 2 * tp[c] + fp[c] + fn[c] else 0.0
        per[labels[c]] = dict(P=round(P[c], 4), R=round(R[c], 4), F1=round(f1[c], 4), support=tp[c] + fn[c], predicted=tp[c] + fp[c])
    union = [c for c in range(K) if tp[c] + fp[c] + fn[c] > 0]
    gold_present = [c for c in range(K) if tp[c] + fn[c] > 0]
    mean = lambda xs, idx: sum(xs[i] for i in idx) / max(1, len(idx))
    return dict(macro_F1=mean(f1, union), macro_P=mean(P, union), macro_R=mean(R, union),
                macro_F1_gold=mean(f1, gold_present), macro_F1_all13=mean(f1, range(K)),
                accuracy=sum(tp) / max(1, len(gold)), per_label=per, n=len(gold))


def score_docs(gold, pred, domain_of, labels):
    """gold/pred: {doc_id: [label ids]}. Returns the scores overall, and for each domain present."""
    out = {}
    for name, keep in [("all", lambda d: True)] + [(dom, (lambda d, dom=dom: domain_of[d] == dom)) for dom in sorted(set(domain_of.values()))]:
        ids = [d for d in gold if keep(d)]
        out[name] = macro_scores([y for d in ids for y in gold[d]], [y for d in ids for y in pred[d]], labels)
    return out


def confusion_counts(gold, pred, labels, top=15):
    c = Counter((labels[g], labels[p]) for d in gold for g, p in zip(gold[d], pred[d]) if g != p)
    return [(f"{a}->{b}", n) for (a, b), n in c.most_common(top)]


def eval_docs(model, docs, C, device, logT=None):
    """(score, details) for the training loop: macro-F1 over all held-out sentences plus the per-domain scores."""
    lps = predict_logprobs(model, docs, C, device)
    labels = C["LABELS"]
    gold = {d["id"]: d["y"] for d in docs}
    pred = {d["id"]: decode_ids(lp, logT, C.get("TRANS_LAMBDA", 0.0)) for d, lp in zip(docs, lps)}
    dom = {d["id"]: d["domain"] for d in docs}
    S = score_docs(gold, pred, dom, labels)
    det = {f"F1_{k}": round(v["macro_F1"], 4) for k, v in S.items() if k != "all"}
    det["acc"] = round(S["all"]["accuracy"], 4)
    return S["all"]["macro_F1"], det


# ----------------------------------------------------------------------------------------------------------------------
# 7. Cross-validation folds, runs, submission
# ----------------------------------------------------------------------------------------------------------------------
def make_folds(docs, k, seed=0):
    """Document-level folds, stratified by domain, balanced by number of sentences (largest documents first)."""
    folds = [[] for _ in range(k)]
    size = [0] * k
    by_dom = defaultdict(list)
    for i, d in enumerate(docs):
        by_dom[d["domain"]].append(i)
    rng = random.Random(seed)
    for dom in sorted(by_dom):
        idx = by_dom[dom][:]
        rng.shuffle(idx)
        idx.sort(key=lambda i: -docs[i]["n"])
        dsize = [0] * k
        for i in idx:
            f = min(range(k), key=lambda j: (dsize[j], size[j], j))
            folds[f].append(i); dsize[f] += docs[i]["n"]; size[f] += docs[i]["n"]
    return folds


def fold_report(docs, folds, labels):
    rows = []
    for f, idx in enumerate(folds):
        c = Counter(y for i in idx for y in docs[i]["y"])
        rows.append(dict(fold=f, docs=len(idx), sentences=sum(docs[i]["n"] for i in idx), domains=dict(Counter(docs[i]["domain"] for i in idx)),
                         labels={labels[k]: c[k] for k in range(len(labels)) if c[k]}))
    return rows


def run_cv(build_fn, docs, folds, C, device, fold_ids=None, test_docs=None, log=print):
    """For every fold: train on the other folds, pick the best epoch on the held-out fold, keep the held-out log-probabilities and (optionally)
    the test log-probabilities of that fold model. build_fn() -> (model, info). Returns dict(per_fold, curves, heldout_lp, test_lp)."""
    fold_ids = list(range(len(folds))) if fold_ids is None else fold_ids
    out = dict(per_fold={}, curves={}, heldout_lp={}, test_lp={})
    for k in fold_ids:
        va = [docs[i] for i in folds[k]]
        tr = [docs[i] for f in range(len(folds)) if f != k for i in folds[f]]
        model, info = build_fn()
        logT = learn_logT(tr, model.n_labels)
        log(f"== fold {k}: train {len(tr)} docs / held-out {len(va)} docs ==")
        res = train_rr(model, tr, C, device, eval_fn=lambda m: eval_docs(m, va, C, device, logT), log=log)
        load_trainable_state(model, res["best_state"])
        out["heldout_lp"][k] = {d["id"]: lp for d, lp in zip(va, predict_logprobs(model, va, C, device))}
        if test_docs is not None:
            out["test_lp"][k] = {d["id"]: lp for d, lp in zip(test_docs, predict_logprobs(model, test_docs, C, device))}
        out["per_fold"][k] = {kk: v for kk, v in res.items() if kk != "best_state"}
        out["per_fold"][k]["technique_info"] = info
        out["curves"][k] = [r.get("heldout_macroF1") for r in res["history"]]
        del model, res
        if device.type == "cuda":
            torch.cuda.empty_cache()
    return out


def choose_epoch(curves, smooth=3):
    """Epoch (1-based) with the best mean held-out score across folds, using a `smooth`-epoch moving average so a flat plateau
    wins over a single noisy peak."""
    n = min(len(c) for c in curves.values())
    mean = [sum(c[i] for c in curves.values()) / len(curves) for i in range(n)]
    sm = [sum(mean[max(0, i - smooth // 2): i + smooth // 2 + 1]) / len(mean[max(0, i - smooth // 2): i + smooth // 2 + 1]) for i in range(n)]
    return int(max(range(n), key=lambda i: sm[i])) + 1, mean


def validate_submission(pred, test_raw, labels):
    """pred: {doc_id: [label strings]}; test_raw: {doc_id: [sentences]}. Raises on any format problem."""
    assert set(pred) == set(test_raw), f"id mismatch: missing {len(set(test_raw) - set(pred))}, extra {len(set(pred) - set(test_raw))}"
    for did, sents in test_raw.items():
        p = pred[did]
        assert isinstance(p, list) and len(p) == len(sents), f"{did}: {len(p)} predictions for {len(sents)} sentences"
        assert all(isinstance(x, str) and x in labels for x in p), f"{did}: a prediction is not an official label string"
    return True


def write_submission(pred_ids, labels, test_raw, out_dir, name):
    """pred_ids: {doc_id: [label ids]} -> prediction.json + a zip with prediction.json at its root. Returns the paths."""
    pred = {did: [labels[i] for i in ids] for did, ids in pred_ids.items()}
    validate_submission(pred, test_raw, labels)
    os.makedirs(out_dir, exist_ok=True)
    jpath = os.path.join(out_dir, f"{name}_prediction.json")
    text = json.dumps(pred, ensure_ascii=False)
    assert "null" not in text, "a null label slipped into the submission"
    with open(jpath, "w", encoding="utf-8") as f:
        f.write(text)
    zpath = os.path.join(out_dir, f"submission_{name}.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jpath, arcname="prediction.json")
    return jpath, zpath

## 8. Adaptation techniques: LoRA, rsLoRA, LoRMA, AdaLoRA, QLoRA, IA3, prompt tuning, prefix tuning, full fine-tuning, frozen

In [7]:
# [P3-PRELUDE:peft]
# ===================== Phase 3 adaptation techniques: frozen, full FT, LoRA, rsLoRA, LoRMA, AdaLoRA, QLoRA, IA3, prompt, prefix =====================
# All of them leave the Phase-1 model code untouched: they wrap/replace nn.Linear modules inside the backbone, or add small trainable tensors.
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------------------------------------------------
# LoRA / rsLoRA / LoRMA
# ----------------------------------------------------------------------------------------------------------------------
class LoRALinear(nn.Module):
    """LoRA (Hu et al. 2022): y = W0 x + b + s * B A x,  A (r, in) random, B (out, r) zeros -> starts exactly at W0.
    s = alpha / r (LoRA)  or  alpha / sqrt(r) (rank-stabilised LoRA, Kalajdzievski 2023: keeps the update size stable when r grows)."""

    def __init__(self, base, r, alpha, dropout=0.0, rank_stabilized=False):
        super().__init__()
        self.base = base
        self.r = r
        self.scale = alpha / math.sqrt(r) if rank_stabilized else alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)


_ROT_IDX = {}


def _row_rotation_index(d, device):
    key = (d, str(device))
    if key not in _ROT_IDX:
        i = torch.arange(d, device=device).unsqueeze(1)
        j = torch.arange(d, device=device).unsqueeze(0)
        _ROT_IDX[key] = (j - i) % d                      # Mpi[i, j] = M[i, (j - i) mod d]  (row i rotated right by i)
    return _ROT_IDX[key]


class LoRMALinear(nn.Module):
    """LoRMA (Bihany, Patel, Modi, ACL Findings 2025): a MULTIPLICATIVE low-rank update of a frozen W0 (out x in).
      "plus" (LoRMA+):  h = ((alpha/r) B A + I) W0 x + b,   A (r, out) random, B (out, r) zeros -> starts at W0. Computed as
                        y0 = W0 x;  y0 + (alpha/r) B (A y0), so the out x out matrix is never formed.
      "pi"   (LoRMA_pi): h = I_pi(B A) W0 x + b, I_pi rotates row i of BA cyclically by i (rank inflation). Init so that I_pi(BA) = I."""

    def __init__(self, base, r, alpha, variant="plus"):
        super().__init__()
        self.base, self.variant, self.r, self.scale = base, variant, r, alpha / r
        d = base.out_features
        if variant == "plus":
            self.A = nn.Parameter(torch.empty(r, d))
            self.B = nn.Parameter(torch.zeros(d, r))
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        elif variant == "pi":
            A = torch.zeros(r, d); A[0, 0] = 1.0
            B = torch.randn(d, r) * (1.0 / math.sqrt(d)); B[:, 0] = 1.0
            self.A, self.B = nn.Parameter(A), nn.Parameter(B)
        else:
            raise ValueError(variant)

    def forward(self, x):
        y0 = F.linear(x, self.base.weight)
        if self.variant == "plus":
            y = y0 + self.scale * F.linear(F.linear(y0, self.A), self.B)
        else:
            M = self.B @ self.A
            Mpi = torch.gather(M, 1, _row_rotation_index(M.shape[0], M.device))
            y = F.linear(y0, Mpi.to(y0.dtype))
        if self.base.bias is not None:
            y = y + self.base.bias
        return y


# ----------------------------------------------------------------------------------------------------------------------
# AdaLoRA (Zhang et al. 2023), simplified
# ----------------------------------------------------------------------------------------------------------------------
class AdaLoRALinear(nn.Module):
    """y = W0 x + b + s * P diag(lam * mask) Q x   (an SVD-shaped update: P (out, r) and Q (r, in) start orthonormal, lam (r) starts at 0).
    The controller below prunes whole (P_i, lam_i, Q_i) triplets by importance during training, moving the rank budget to the layers that
    need it. Simplification: a pruned triplet stays pruned."""

    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        out, inn = base.out_features, base.in_features
        P = torch.linalg.qr(torch.randn(out, r))[0]                      # orthonormal columns
        Q = torch.linalg.qr(torch.randn(inn, r))[0].t().contiguous()     # orthonormal rows
        self.P, self.Q = nn.Parameter(P), nn.Parameter(Q)
        self.lam = nn.Parameter(torch.zeros(r))
        self.register_buffer("ada_mask", torch.ones(r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        z = F.linear(self.drop(x), self.Q) * (self.lam * self.ada_mask)
        return self.base(x) + self.scale * F.linear(z, self.P)

    def orth_penalty(self):
        eye = torch.eye(self.r, device=self.P.device)
        return ((self.P.t() @ self.P - eye) ** 2).sum() + ((self.Q @ self.Q.t() - eye) ** 2).sum()


class AdaLoRAController:
    """Importance score of a triplet i: s(lam_i) + mean_j s(P_ji) + mean_j s(Q_ij), with s(w) = smoothed |w * grad(w)| times its
    smoothed uncertainty (paper, eq. 8-10). Budget schedule: full budget for the first t_i steps, then a cubic decrease to the target
    budget, then fixed for the last t_f steps. Also adds the orthogonality penalty to the loss."""

    def __init__(self, modules, target_rank, ti_frac=0.1, tf_frac=0.2, delta_t=10, beta=0.85, orth_weight=0.1):
        self.mods = modules
        self.r0 = modules[0].r
        self.b0 = len(modules) * self.r0
        self.bT = int(round(len(modules) * target_rank))
        self.ti_frac, self.tf_frac, self.dt, self.beta, self.orth = ti_frac, tf_frac, delta_t, beta, orth_weight
        z = lambda p: torch.zeros_like(p.data)
        self.st = [{n: (z(getattr(m, n)), z(getattr(m, n))) for n in ("lam", "P", "Q")} for m in modules]

    def extra_loss(self):
        return self.orth * sum(m.orth_penalty() for m in self.mods) / len(self.mods)

    def budget(self, step, total):
        ti, tf = int(self.ti_frac * total), int(self.tf_frac * total)
        if step < ti:
            return self.b0
        if step > total - tf:
            return self.bT
        frac = (step - ti) / max(1, total - ti - tf)
        return int(round(self.bT + (self.b0 - self.bT) * (1 - frac) ** 3))

    @torch.no_grad()
    def on_before_step(self, step, total):
        b = self.beta
        for m, st in zip(self.mods, self.st):
            for n, (ibar, u) in list(st.items()):
                p = getattr(m, n)
                if ibar.device != p.device:                       # the statistics were created on the CPU, the model may have moved since
                    ibar, u = ibar.to(p.device), u.to(p.device); st[n] = (ibar, u)
                if p.grad is None:
                    continue
                imp = (p.data * p.grad).abs()
                ibar.mul_(b).add_(imp, alpha=1 - b)
                u.mul_((b)).add_((imp - ibar).abs(), alpha=1 - b)
        tf = int(self.tf_frac * total)
        if step % self.dt == 0 and step <= total - tf:
            scores = []
            for m, st in zip(self.mods, self.st):
                s = lambda n: st[n][0] * st[n][1]
                scores.append(s("lam") + s("P").mean(0) + s("Q").mean(1))
            allsc = torch.cat(scores)
            keep = min(max(self.budget(step, total), 1), allsc.numel())
            thr = torch.sort(allsc, descending=True).values[keep - 1]
            for m, sc in zip(self.mods, scores):
                m.ada_mask.copy_((sc >= thr).float() * m.ada_mask)       # a pruned triplet stays pruned

    def rank_report(self):
        return [int(m.ada_mask.sum().item()) for m in self.mods]


# ----------------------------------------------------------------------------------------------------------------------
# QLoRA (Dettmers et al. 2023): frozen 4-bit NormalFloat base weights (+ double quantisation of the block constants) with LoRA on top
# ----------------------------------------------------------------------------------------------------------------------
NF4_TABLE = torch.tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635, -0.18477343022823334,
                          -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
                          0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0])


class NF4Linear(nn.Module):
    """A frozen Linear stored as 4-bit NF4 codes (two per byte) with one absmax constant per `block` weights; the constants are themselves
    stored in 8 bit (double quantisation). The weight is rebuilt on the fly in every forward pass (pure PyTorch: no bitsandbytes)."""

    def __init__(self, base, block=64, double_quant=True, dq_block=256):
        super().__init__()
        self.in_features, self.out_features, self.block, self.double_quant = base.in_features, base.out_features, block, double_quant
        W = base.weight.detach().float().reshape(-1)
        self.numel = W.numel()
        Wp = torch.cat([W, W.new_zeros((-self.numel) % block)])
        blocks = Wp.view(-1, block)
        absmax = blocks.abs().amax(1).clamp(min=1e-8)
        flat = (blocks / absmax[:, None]).reshape(-1)
        codes = torch.empty(flat.numel(), dtype=torch.uint8)
        for i in range(0, flat.numel(), 1 << 21):
            ch = flat[i:i + (1 << 21)]
            codes[i:i + ch.numel()] = (ch.unsqueeze(1) - NF4_TABLE).abs().argmin(1).to(torch.uint8)
        self.register_buffer("packed", (codes[0::2] << 4) | codes[1::2])
        self.register_buffer("nf4", NF4_TABLE.clone(), persistent=False)
        self.n_blocks = absmax.numel()
        if double_quant:
            mean = absmax.mean()
            am = absmax - mean
            amp = torch.cat([am, am.new_zeros((-am.numel()) % dq_block)]).view(-1, dq_block)
            sc = amp.abs().amax(1).clamp(min=1e-12)
            self.register_buffer("am_q", torch.round(amp / sc[:, None] * 127).to(torch.int8))
            self.register_buffer("am_scale", sc)
            self.register_buffer("am_mean", mean.reshape(1))
        else:
            self.register_buffer("absmax", absmax)
        self.register_buffer("bias", base.bias.detach().clone() if base.bias is not None else None)

    def _absmax(self):
        if self.double_quant:
            return (self.am_q.float() / 127 * self.am_scale[:, None]).reshape(-1)[:self.n_blocks] + self.am_mean
        return self.absmax

    @torch.no_grad()
    def dequant(self, dtype=torch.float32):
        codes = torch.stack([(self.packed >> 4).long(), (self.packed & 0x0F).long()], 1).reshape(-1)
        vals = self.nf4[codes].view(-1, self.block) * self._absmax()[:, None]
        return vals.reshape(-1)[:self.numel].view(self.out_features, self.in_features).to(dtype)

    def forward(self, x):
        W = self.dequant(x.dtype)
        return F.linear(x, W, None if self.bias is None else self.bias.to(x.dtype))


# ----------------------------------------------------------------------------------------------------------------------
# IA3 (Liu et al. 2022): learned per-channel rescaling vectors, initialised to 1
# ----------------------------------------------------------------------------------------------------------------------
class IA3Out(nn.Module):
    """y = l * Linear(x): rescales the OUTPUT channels (used on the key and value projections)."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        return self.base(x) * self.l


class IA3In(nn.Module):
    """y = Linear(l * x): rescales the INPUT channels (used on the feed-forward down-projection, i.e. the hidden activation)."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.l)


# ----------------------------------------------------------------------------------------------------------------------
# Applying a technique
# ----------------------------------------------------------------------------------------------------------------------
def _swap_linears(root, fn):
    """fn(qualified_name, nn.Linear) -> replacement module or None. Returns the names that were replaced."""
    done = []
    for qual, mod in list(root.named_modules()):
        for cname, child in list(mod.named_children()):
            if isinstance(child, nn.Linear):
                full = f"{qual}.{cname}" if qual else cname
                new = fn(full, child)
                if new is not None:
                    setattr(mod, cname, new); done.append(full)
    return done


def _hit(full, suffixes):
    return any(full.endswith(s) for s in suffixes)


def size_mb(model):
    return sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6 + sum(b.numel() * b.element_size() for b in model.buffers()) / 1e6


def apply_technique(model, C, log=print):
    """Freezes/unfreezes the backbone and inserts modules according to C['TECHNIQUE']; the sentence head is always trainable."""
    tech, bb = C["TECHNIQUE"], model.backbone
    info = dict(technique=tech)
    for p in bb.parameters():
        p.requires_grad = False
    T = C.get("PEFT_TARGETS", [])
    if tech == "frozen":
        pass                                                           # Phase-1 weights untouched: only the head learns (control)
    elif tech == "full_ft":
        for p in bb.parameters():
            p.requires_grad = True
    elif tech in ("lora", "rslora"):
        done = _swap_linears(bb, lambda n, l: LoRALinear(l, C["RANK"], C["ALPHA"], C.get("LORA_DROPOUT", 0.0), tech == "rslora") if _hit(n, T) else None)
        assert done, f"no linear layer matches PEFT_TARGETS={T}"
        info.update(targets=len(done), rank=C["RANK"], alpha=C["ALPHA"], scale=(C["ALPHA"] / math.sqrt(C["RANK"]) if tech == "rslora" else C["ALPHA"] / C["RANK"]))
    elif tech == "lorma":
        done = _swap_linears(bb, lambda n, l: LoRMALinear(l, C["RANK"], C["ALPHA"], C["LORMA_VARIANT"]) if _hit(n, T) else None)
        assert done, f"no linear layer matches PEFT_TARGETS={T}"
        info.update(targets=len(done), rank=C["RANK"], alpha=C["ALPHA"], variant=C["LORMA_VARIANT"])
    elif tech == "adalora":
        mods = []

        def mk(n, l):
            if _hit(n, T):
                m = AdaLoRALinear(l, C["ADALORA_INIT_RANK"], C["ALPHA"], C.get("LORA_DROPOUT", 0.0)); mods.append(m); return m
        done = _swap_linears(bb, mk)
        assert done, f"no linear layer matches PEFT_TARGETS={T}"
        ctrl = AdaLoRAController(mods, C["ADALORA_TARGET_RANK"], delta_t=C.get("ADALORA_DELTA_T", 10), orth_weight=C.get("ADALORA_ORTH", 0.1))
        model.extra_loss, model.on_before_step, model.ada_controller = ctrl.extra_loss, ctrl.on_before_step, ctrl
        info.update(targets=len(done), init_rank=C["ADALORA_INIT_RANK"], target_rank=C["ADALORA_TARGET_RANK"])
    elif tech == "qlora":
        blocks = bb.layer_list()
        n_q, n_l = [0], [0]

        def mk(n, l):
            q = NF4Linear(l, C.get("NF4_BLOCK", 64), C.get("NF4_DOUBLE_QUANT", True))
            n_q[0] += 1
            if _hit(n, T):
                n_l[0] += 1
                return LoRALinear(q, C["RANK"], C["ALPHA"], C.get("LORA_DROPOUT", 0.0))
            return q
        for blk in blocks:
            _swap_linears(blk, mk)
        assert n_l[0], f"no linear layer matches PEFT_TARGETS={T}"
        info.update(quantized_linears=n_q[0], lora_targets=n_l[0], rank=C["RANK"], alpha=C["ALPHA"])
    elif tech == "ia3":
        n_out, n_in = [0], [0]

        def mk(n, l):
            if _hit(n, C["IA3_OUT"]):
                n_out[0] += 1; return IA3Out(l)
            if _hit(n, C["IA3_IN"]):
                n_in[0] += 1; return IA3In(l)
        _swap_linears(bb, mk)
        assert n_out[0] + n_in[0], f"no linear layer matches IA3_OUT={C['IA3_OUT']} / IA3_IN={C['IA3_IN']}"
        info.update(out_scaled=n_out[0], in_scaled=n_in[0])
    elif tech == "prompt":
        P = C["PROMPT_LEN"]
        emb = bb.embedding_weight()
        idx = torch.randint(100, emb.shape[0], (P,), generator=torch.Generator().manual_seed(0))
        model.soft_prompt = nn.Parameter(emb[idx].detach().clone().float())
        info.update(prompt_len=P)
    elif tech == "prefix":
        bb.enable_prefix(C["PREFIX_LEN"], 0)
        info.update(prefix_len=C["PREFIX_LEN"])
    else:
        raise ValueError(f"unknown TECHNIQUE {tech}")
    for p in model.head_parameters():
        p.requires_grad = True
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    orig = bb.original_param_count
    info.update(params_total=total, params_trainable=trainable, params_backbone_original=orig,
                trainable_pct_of_backbone=round(100.0 * trainable / orig, 4), model_size_mb=round(size_mb(model), 1))
    log(f"Technique {tech}: trainable {trainable:,} of {total:,} parameters ({info['trainable_pct_of_backbone']}% of the Phase-1 model's {orig:,}); "
        f"stored size {info['model_size_mb']} MB")
    return info

## 9. Phase-3 data

In [8]:
# [P3-PRELUDE:data]
P3 = "Phase_3_RR"
def load_json(p):
    with open(p, encoding="utf-8") as f:          # plain json: the label string "None" must stay a string
        return json.load(f)

LABELS = labels_from_vocab(load_json(get_dataset_file(f"{P3}/Train/label_vocab.json")))
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
N_LABELS = len(LABELS)
LABELLED_RAW, TEST_RAW = [], {}
for dom in ("CL", "IT"):
    for sp in ("train", "dev"):
        ds = as_doc_list(load_json(get_dataset_file(f"{P3}/Train/{dom}_{sp}.json")), dom)
        for d in ds:
            d["split"] = sp
        LABELLED_RAW += ds
    TEST_RAW[dom] = load_json(get_dataset_file(f"{P3}/Test/{dom}_test.json"))
TEST_ALL = {k: v for dom in TEST_RAW for k, v in TEST_RAW[dom].items()}
assert len(TEST_ALL) == sum(len(v) for v in TEST_RAW.values()), "a test document id occurs in both domains"
assert len({d["id"] for d in LABELLED_RAW}) == len(LABELLED_RAW), "duplicate labelled ids"
assert not ({d["id"] for d in LABELLED_RAW} & set(TEST_ALL)), "a test id is also a labelled id"
for d in LABELLED_RAW:
    assert len(d["text"]) == len(d["labels"]) and all(l in LABEL2ID for l in d["labels"])
if QUICK_TEST:                                   # tiny, fast check: 4 documents per domain, first 40 sentences each, 1 test document per domain
    keep = []
    for dom in ("CL", "IT"):
        for d in sorted([x for x in LABELLED_RAW if x["domain"] == dom], key=lambda x: len(x["text"]))[:4]:
            d["text"], d["labels"] = d["text"][:40], d["labels"][:40]; keep.append(d)
    LABELLED_RAW = keep
    TEST_RAW = {dom: {k: v[:30] for k, v in list(TEST_RAW[dom].items())[:1]} for dom in TEST_RAW}
    TEST_ALL = {k: v for dom in TEST_RAW for k, v in TEST_RAW[dom].items()}
print("labels:", LABELS)
print("labelled documents:", Counter((d["domain"], d["split"]) for d in LABELLED_RAW))
print("test documents:", {dom: (len(v), sum(len(x) for x in v.values())) for dom, v in TEST_RAW.items()}, "(documents, sentences)")

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

labels: ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']
labelled documents: Counter({('CL', 'train'): 40, ('IT', 'train'): 40, ('CL', 'dev'): 5, ('IT', 'dev'): 5})
test documents: {'CL': (5, 1339), 'IT': (5, 858)} (documents, sentences)


## 10. Tokenize the documents, cross-validation folds

In [9]:
# [P3-PRELUDE:tokenize]
t0 = time.time()
LABELLED, tinfo = encode_docs(bpe_tokenizer, LABELLED_RAW, LABEL2ID, BASE["MAX_SENT_TOKENS"])
TEST_DOCS = []
for dom in ("CL", "IT"):
    td, _ = encode_docs(bpe_tokenizer, as_doc_list(TEST_RAW[dom], dom), LABEL2ID, BASE["MAX_SENT_TOKENS"])
    TEST_DOCS += td
DOMAIN_OF = {d["id"]: d["domain"] for d in LABELLED + TEST_DOCS}
_q = lambda a, p: sorted(a)[min(len(a) - 1, int(p * len(a)))]
_st = [len(s) for d in LABELLED for s in d["sents"]]
_dt = [sum(len(s) for s in d["sents"]) for d in LABELLED]
print(f"{tinfo['sentences']} sentences tokenized, {tinfo['truncated']} cut at MAX_SENT_TOKENS={BASE['MAX_SENT_TOKENS']}")
print(f"tokens/sentence: median {_q(_st,.5)}, p95 {_q(_st,.95)}, max {max(_st)} | tokens/document: median {_q(_dt,.5)}, p90 {_q(_dt,.9)}, max {max(_dt)}; "
      f"{100*sum(1 for t in _dt if t <= 2048)/len(_dt):.0f}% of documents fit in 2048 tokens")
assert max(_st) <= BASE["WINDOW"], "WINDOW is smaller than the longest sentence: lower MAX_SENT_TOKENS"
# cross-validation folds: whole documents, balanced by domain and by size
FOLDS = make_folds(LABELLED, K_FOLDS, FOLD_SEED)
for r in fold_report(LABELLED, FOLDS, LABELS):
    print(f"  fold {r['fold']}: {r['docs']} docs {r['domains']}, {r['sentences']} sentences")
SAN_DOCS = {d["id"]: {"ids": [t for s in d["sents"] for t in s]} for d in LABELLED[:8]}   # for the Phase-1 sanity check
print(f"[{time.time()-t0:.0f}s]")

18987 sentences tokenized, 36 cut at MAX_SENT_TOKENS=256
tokens/sentence: median 30, p95 84, max 256 | tokens/document: median 4908, p90 16334, max 66543; 26% of documents fit in 2048 tokens
  fold 0: 16 docs {'CL': 7, 'IT': 9}, 3789 sentences
  fold 1: 17 docs {'CL': 8, 'IT': 9}, 3800 sentences
  fold 2: 19 docs {'CL': 10, 'IT': 9}, 3790 sentences
  fold 3: 19 docs {'CL': 10, 'IT': 9}, 3798 sentences
  fold 4: 19 docs {'CL': 10, 'IT': 9}, 3810 sentences
[1s]


## 11. Load the Phase-1 checkpoint, check it, define how a model is built

In [10]:
# [P3-PRELUDE:build]
t0 = time.time()
_ck = torch.load(fetch_model_file(PHASE1_CHECKPOINT), map_location="cpu", mmap=True)
PHASE1_STATE = _ck["model"] if "model" in _ck else _ck
del _ck
_lm = build_phase1_lm()
PHASE1_PARAMS = sum(p.numel() for p in _lm.parameters())
SANITY_LOSS = lm_sanity_loss(_lm, SAN_DOCS, device)
print(f"Phase-1 model {MODEL_KEY}: {PHASE1_PARAMS:,} parameters; next-token loss on the case texts = {SANITY_LOSS:.3f} nats "
      f"(far below {math.log(VOCAB):.1f} = trained weights loaded; about {math.log(VOCAB):.1f} = random weights) [{time.time()-t0:.0f}s]")
assert SANITY_LOSS < 8.0, "the Phase-1 weights do not seem to be loaded correctly"
del _lm; gc.collect(); torch.cuda.empty_cache() if device.type == "cuda" else None


def new_backbone(lm):
    return DecoderBackbone(lm, bidirectional=False)


def build_model(C, log=print):
    """Phase-1 model -> backbone -> sentence model (pooling + BiLSTM head) -> adaptation technique."""
    lm = build_phase1_lm()
    bb = new_backbone(lm)
    model = RRModel(bb, bb.d_model, N_LABELS, dict(C, POOL=C["POOL"]))
    info = apply_technique(model, C, log)
    return model.to(device), info


def free():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

phase1_decoder_only_gqa_kvcache_full_lay(…): reconstructing file:   0%|          |  0.00B / 1.33GB            

phase1_decoder_only_gqa_kvcache_full_lay(…): downloading bytes:           |  0.00B            

(found phase1_decoder_only_gqa_kvcache_full_layernorm_pre_adamw_rope_ctx2048_160Mw_a1bpe_160Mw_1ep_checkpoint.pt in the old repo kavincg23/cs781-checkpoints)
Phase-1 model gqa160: 111,095,872 parameters; next-token loss on the case texts = 3.871 nats (far below 10.6 = trained weights loaded; about 10.6 = random weights) [14s]


## 12. Cross-validation of every setup (resumable; one process per GPU when several GPUs are available)

In [11]:
# [P3-PRELUDE:cvdefs]
import hashlib, pickle
RES_DIR = f"{WORK}/p3res_{SWEEP_NAME}"
os.makedirs(RES_DIR, exist_ok=True)
SWEEP_START = float(os.environ.get('P3_T0', time.time()))
ID2IDX = {d["id"]: i for i, d in enumerate(LABELLED)}


def make_C(overrides):
    C = dict(BASE, **overrides)
    C["LABELS"] = LABELS
    if C["TECHNIQUE"] == "full_ft" and "LR" not in overrides:
        C["LR"] = FULL["LR"]
    return C


def groups_of(C):
    allx = list(range(len(LABELLED)))
    if C["DOMAIN_MODE"] == "separate":
        return {dom: [i for i in allx if LABELLED[i]["domain"] == dom] for dom in ("CL", "IT")}
    return {"all": allx}


def cfg_key(C, group):
    c = {k: v for k, v in C.items() if k != "LABELS"}
    return hashlib.md5(json.dumps([c, group, K_FOLDS, FOLD_SEED, QUICK_TEST, len(LABELLED)], sort_keys=True, default=str).encode()).hexdigest()[:10]


def cache_name(setup, group, k):
    return f"p3res_{SWEEP_NAME}__{setup}__{group}__f{k}.pt"


def cache_load(setup, group, k, key):
    name = cache_name(setup, group, k)
    try:
        path = fetch_model_file(name, f"{RES_DIR}/{name}")
        r = torch.load(path, map_location="cpu", weights_only=False)
        return r if r.get("key") == key else None
    except Exception:
        return None


def cache_save(setup, group, k, r):
    path = f"{RES_DIR}/{cache_name(setup, group, k)}"
    torch.save(r, path)
    upload_model_file(path, cache_name(setup, group, k))


def run_fold(setup, C, group, idx, k, key):
    cached = cache_load(setup, group, k, key)
    if cached is not None:
        print(f"  [{setup} / {group} / fold {k}] cached ({len(cached['curve'])} epochs, best {max(cached['curve']):.4f})")
        return cached
    gset = set(idx)
    va = [LABELLED[i] for i in FOLDS[k] if i in gset]
    tr = [LABELLED[i] for f in range(K_FOLDS) if f != k for i in FOLDS[f] if i in gset]
    if not va or not tr:
        return None
    model, info = build_model(C, log=lambda *a: None)
    logT = learn_logT(tr, N_LABELS)
    store = {}

    def eval_fn(m):
        lps = predict_logprobs(m, va, C, device)
        store[len(store) + 1] = {d["id"]: lp.clone() for d, lp in zip(va, lps)}
        gold = {d["id"]: d["y"] for d in va}
        pred = {d["id"]: lp.argmax(-1).tolist() for d, lp in zip(va, lps)}
        S = score_docs(gold, pred, {d["id"]: d["domain"] for d in va}, LABELS)
        return S["all"]["macro_F1"], {f"F1_{n}": round(v["macro_F1"], 4) for n, v in S.items() if n != "all"}

    print(f"== {setup} / {group} / fold {k}: train {len(tr)} docs ({sum(d['n'] for d in tr)} sentences), held-out {len(va)} docs ({sum(d['n'] for d in va)} sentences)")
    res = train_rr(model, tr, C, device, eval_fn=eval_fn, epochs=C["EPOCHS"], keep_best=False)
    r = dict(key=key, curve=[h["heldout_macroF1"] for h in res["history"]], lps=store, logT=logT, info=info, seconds=res["train_seconds"],
             peak_mem_gb=res["peak_mem_gb"], tokens_per_sec=res["tokens_per_sec"], history=res["history"], val_ids=[d["id"] for d in va])
    if hasattr(model, "ada_controller"):
        r["adalora_ranks"] = model.ada_controller.rank_report()
    cache_save(setup, group, k, r)
    del model, res
    free()
    return r


def summarize(setup, C, folds_res, lam=None):
    """folds_res: {group: {k: fold result}}. Chooses one epoch per group from the flat part of the mean held-out curve, then scores the
    held-out predictions of that epoch (all folds together) overall, for CL and for IT."""
    chosen, curves_mean = {}, {}
    for g, fr in folds_res.items():
        chosen[g], curves_mean[g] = choose_epoch({k: r["curve"] for k, r in fr.items()})
    out = {}
    for lam_ in ([lam] if lam is not None else TRANS_LAMBDAS):
        gold, pred = {}, {}
        for g, fr in folds_res.items():
            E = min(chosen[g], min(len(r["curve"]) for r in fr.values()))
            for k, r in fr.items():
                for did, lp in r["lps"][E].items():
                    gold[did] = LABELLED[ID2IDX[did]]["y"]
                    pred[did] = decode_ids(lp, r["logT"], lam_)
        out[lam_] = (score_docs(gold, pred, DOMAIN_OF, LABELS), gold, pred)
    return chosen, curves_mean, out




def make_jobs():
    """Every (setup, group, fold) of the sweep, in a fixed order. Workers take every N-th job."""
    jobs = []
    for name, ov in SWEEP:
        C = make_C(ov)
        for g, idx in groups_of(C).items():
            key = cfg_key(C, g)
            for k in range(K_FOLDS):
                jobs.append((name, C, g, idx, k, key))
    return jobs

In [12]:
import re, threading
PRELUDE_ORDER = ["settings", "setup", "bpe", "model", "wrapper", "core", "peft", "data", "tokenize", "build", "cvdefs"]
WORKER_HEADER = """import os, sys, types
_m = types.ModuleType("kaggle_secrets")
class UserSecretsClient:
    def get_secret(self, label):
        return os.environ["P3_SECRET_" + label]
_m.UserSecretsClient = UserSecretsClient
sys.modules["kaggle_secrets"] = _m
"""
WORKER_MAIN = """
_W, _N = int(os.environ["P3_WORKER"]), int(os.environ["P3_NWORK"])
for _ji, (_name, _C, _g, _idx, _k, _key) in enumerate(make_jobs()):
    if _ji % _N != _W:
        continue
    if (time.time() - SWEEP_START) / 3600 > MAX_SWEEP_HOURS and cache_load(_name, _g, _k, _key) is None:
        print(f"MAX_SWEEP_HOURS reached: not starting {_name}/{_g}/fold {_k}"); continue
    run_fold(_name, _C, _g, _idx, _k, _key)
print("WORKER DONE")
"""


def launch_workers(n):
    """One process per GPU. Each runs the same notebook cells (taken from this session's own cell history) and trains its share of the
    folds; results go to the same cache that the loop below reads, so nothing else changes."""
    hist = get_ipython().user_ns["In"]
    src = {}
    for text in hist:
        m = re.match(r"# \[P3-PRELUDE:(\w+)\]", text or "")
        if m:
            src[m.group(1)] = text                                  # the latest run of each cell
    missing = [k for k in PRELUDE_ORDER if k not in src]
    assert not missing, f"cells not run in this session: {missing}"
    body = "\n\n".join("\n".join(l for l in src[k].splitlines() if not (l.lstrip().startswith(("!", "%")) or "get_ipython()" in l)) for k in PRELUDE_ORDER)
    path = f"{WORK}/p3_worker.py"
    with open(path, "w") as f:
        f.write(WORKER_HEADER + body + "\n" + WORKER_MAIN)
    procs = []
    for i in range(n):
        env = dict(os.environ, P3_WORKER=str(i), P3_NWORK=str(n), P3_T0=str(SWEEP_START), CUDA_VISIBLE_DEVICES=str(i), PYTHONUNBUFFERED="1",
                   P3_SECRET_HF=HF_TOKEN, P3_SECRET_HF_WRITE=HF_WRITE_TOKEN)
        procs.append(subprocess.Popen([sys.executable, path], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1))

    lock = threading.Lock()

    def pump(i, p):
        for line in p.stdout:
            with lock:
                print(f"[gpu{i}] {line.rstrip()}", flush=True)
    ths = [threading.Thread(target=pump, args=(i, p), daemon=True) for i, p in enumerate(procs)]
    for t in ths:
        t.start()
    for p in procs:
        p.wait()
    for t in ths:
        t.join(timeout=5)
    bad = [i for i, p in enumerate(procs) if p.returncode != 0]
    if bad:
        print(f"WARNING: worker(s) {bad} ended with an error (see the lines above). Folds they did not finish are trained one by one below.")
    return not bad


N_WORKERS = WORKERS_OVERRIDE or (min(2, torch.cuda.device_count()) if USE_ALL_GPUS else 1)
if N_WORKERS > 1 and not os.environ.get("P3_WORKER"):
    print(f"splitting the cross-validation folds over {N_WORKERS} processes (one per GPU)")
    t_w = time.time()
    launch_workers(N_WORKERS)
    print(f"workers finished in {(time.time()-t_w)/60:.0f} min; collecting the results")

RESULTS = {}
t_all = time.time()
for name, ov in SWEEP:
    C = make_C(ov)
    folds_res = {}
    stop = False
    for g, idx in groups_of(C).items():
        key = cfg_key(C, g)
        folds_res[g] = {}
        for k in range(K_FOLDS):
            if (time.time() - SWEEP_START) / 3600 > MAX_SWEEP_HOURS and cache_load(name, g, k, key) is None:
                print(f"MAX_SWEEP_HOURS reached: not starting {name}/{g}/fold {k}")
                stop = True
                break
            r = run_fold(name, C, g, idx, k, key)
            if r is not None:
                folds_res[g][k] = r
        if stop:
            break
    if stop or any(len(fr) == 0 for fr in folds_res.values()):
        print(f"{name}: incomplete, skipped in the ranking"); continue
    chosen, curves_mean, by_lam = summarize(name, C, folds_res)
    best_lam = max(by_lam, key=lambda l: by_lam[l][0]["all"]["macro_F1"])
    S = by_lam[best_lam][0]
    infos = [r["info"] for fr in folds_res.values() for r in fr.values()]
    RESULTS[name] = dict(
        overrides=ov, C=C, chosen_epoch=chosen, mean_curve=curves_mean, best_lambda=best_lam,
        scores={lam_: {n: {k_: v for k_, v in sc.items() if k_ != "per_label"} for n, sc in by_lam[lam_][0].items()} for lam_ in by_lam},
        per_label={n: sc["per_label"] for n, sc in S.items()},
        confusion=confusion_counts(by_lam[best_lam][1], by_lam[best_lam][2], LABELS),
        trainable=infos[0]["params_trainable"], total=infos[0]["params_total"], pct_backbone=infos[0]["trainable_pct_of_backbone"],
        size_mb=infos[0]["model_size_mb"], peak_mem_gb=max([r["peak_mem_gb"] or 0 for fr in folds_res.values() for r in fr.values()]),
        minutes_per_fold=round(sum(r["seconds"] for fr in folds_res.values() for r in fr.values()) / 60 / max(1, sum(len(fr) for fr in folds_res.values())), 1),
        fold_curves={g: {k: r["curve"] for k, r in fr.items()} for g, fr in folds_res.items()},
        adalora_ranks=[r.get("adalora_ranks") for fr in folds_res.values() for r in fr.values()][:1])
    a, c, i = S["all"], S.get("CL", {}), S.get("IT", {})
    print(f"### {name}: epoch {chosen} | macro-F1 all {a['macro_F1']:.4f} (P {a['macro_P']:.3f} R {a['macro_R']:.3f}) "
          f"CL {c.get('macro_F1', float('nan')):.4f} IT {i.get('macro_F1', float('nan')):.4f} | lambda {best_lam} | {RESULTS[name]['minutes_per_fold']} min/fold")
    with open(f"{RES_DIR}/results_{SWEEP_NAME}.json", "w") as f:
        json.dump(RESULTS, f, indent=1, default=str)
    FOLDS_CACHE = None
print(f"cross-validation done in {(time.time()-t_all)/60:.0f} min")

  [lora_pooled / all / fold 0] cached (12 epochs, best 0.5340)


  [lora_pooled / all / fold 1] cached (12 epochs, best 0.4579)


  [lora_pooled / all / fold 2] cached (12 epochs, best 0.4803)


  [lora_pooled / all / fold 3] cached (12 epochs, best 0.4756)


  [lora_pooled / all / fold 4] cached (12 epochs, best 0.4843)
### lora_pooled: epoch {'all': 10} | macro-F1 all 0.4392 (P 0.478 R 0.424) CL 0.4405 IT 0.4700 | lambda 0.0 | 5.6 min/fold


  [lora_separate / CL / fold 0] cached (12 epochs, best 0.4788)


... (155 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded p3res_p3s_gqa160__frozen_head_only__all__f4.pt to bucket kavincg23/cs781-models
### frozen_head_only: epoch {'all': 8} | macro-F1 all 0.4075 (P 0.429 R 0.397) CL 0.4004 IT 0.4517 | lambda 0.0 | 2.6 min/fold
cross-validation done in 197 min


## 13. Ranking, per-label scores, decoding ablation

In [13]:
rows = sorted(RESULTS.items(), key=lambda kv: -kv[1]["scores"][kv[1]["best_lambda"]]["all"]["macro_F1"])
print(f"{'setup':26s} {'F1 all':>7s} {'CL':>7s} {'IT':>7s} {'P':>6s} {'R':>6s} {'acc':>6s} {'ep':>4s} {'lam':>4s} {'train M':>8s} {'%bb':>6s} {'MB':>7s} {'GB':>5s} {'min/fold':>8s}")
for n, r in rows:
    s = r["scores"][r["best_lambda"]]
    print(f"{n:26s} {s['all']['macro_F1']:7.4f} {s.get('CL', {}).get('macro_F1', float('nan')):7.4f} {s.get('IT', {}).get('macro_F1', float('nan')):7.4f} "
          f"{s['all']['macro_P']:6.3f} {s['all']['macro_R']:6.3f} {s['all']['accuracy']:6.3f} {str(list(r['chosen_epoch'].values())[0]):>4s} {r['best_lambda']:4} "
          f"{r['trainable']/1e6:8.2f} {r['pct_backbone']:6.2f} {r['size_mb']:7.0f} {r['peak_mem_gb']:5.1f} {r['minutes_per_fold']:8.1f}")
if rows:
    best, rb = rows[0]
    print(f"\nBest: {best}. Per-label F1 (cross-validated, all documents together):")
    for l in LABELS:
        pl = rb["per_label"]["all"][l]
        print(f"  {l:24s} F1 {pl['F1']:.3f}  P {pl['P']:.3f}  R {pl['R']:.3f}  support {pl['support']}")
    print("most frequent confusions (gold->predicted):", rb["confusion"][:10])
    print("\nDecoding ablation for the best setup (macro-F1 all / CL / IT):")
    for lam_, s in rb["scores"].items():
        print(f"  lambda {lam_}: {s['all']['macro_F1']:.4f} / {s.get('CL', {}).get('macro_F1', float('nan')):.4f} / {s.get('IT', {}).get('macro_F1', float('nan')):.4f}")

setup                       F1 all      CL      IT      P      R    acc   ep  lam  train M    %bb      MB    GB min/fold
rslora                      0.4450  0.4430  0.4472  0.477  0.433  0.583   10  0.0     4.32   3.89     340   1.5      5.6
full_ft                     0.4417  0.4419  0.4771  0.468  0.434  0.582    9  0.0    84.02  75.63     337   2.8      6.3
lora_pooled                 0.4392  0.4405  0.4700  0.478  0.424  0.577   10  0.0     4.32   3.89     340   1.5      5.6
lorma                       0.4392  0.4553  0.4623  0.477  0.422  0.575    9  0.3     4.18   3.76     339   1.5      5.5
lora_separate               0.4378  0.4048  0.4731  0.471  0.422  0.569   12  0.0     4.32   3.89     340   1.5      2.8
qlora                       0.4368  0.4351  0.4720  0.476  0.421  0.573   11  0.0     4.32   3.89     167   1.4      6.6
adalora                     0.4152  0.4181  0.4543  0.457  0.398  0.551   10  0.0     4.64   4.18     341   1.6      6.5
prompt                      0.41

## 14. Refit the best setups on all labelled documents and write the submission zips

In [14]:
SUB_DIR = f"{WORK}/p3_submissions_{SWEEP_NAME}"
os.makedirs(SUB_DIR, exist_ok=True)
ranked = [n for n, _ in rows]
to_refit = list(dict.fromkeys(ranked[:REFIT_TOP_K] + [n for n in REFIT_ALWAYS if n in RESULTS]))
print("refit on all labelled documents:", to_refit)
TEST_LP = {}
for name in to_refit:
    r = RESULTS[name]; C = r["C"]
    pred_ids, test_lp = {}, {}
    for g, idx in groups_of(C).items():
        tr = [LABELLED[i] for i in idx]
        E = r["chosen_epoch"][g]
        print(f"== refit {name} / {g}: {len(tr)} documents, {E} epochs (the learning-rate decay ends at epoch {E})")
        model, info = build_model(C, log=lambda *a: None)
        res = train_rr(model, tr, C, device, eval_fn=None, epochs=E, keep_best=False)
        targets = [d for d in TEST_DOCS if g == "all" or d["domain"] == g]
        logT = learn_logT(tr, N_LABELS)
        for d, lp in zip(targets, predict_logprobs(model, targets, C, device)):
            test_lp[d["id"]] = lp
            pred_ids[d["id"]] = decode_ids(lp, logT, r["best_lambda"])
        del model, res; free()
    jpath, zpath = write_submission(pred_ids, LABELS, TEST_ALL, SUB_DIR, f"{SWEEP_NAME}_{name}")
    TEST_LP[name] = test_lp
    dist = Counter(LABELS[i] for ids in pred_ids.values() for i in ids)
    print(f"   wrote {zpath}; predicted label counts: {dict(dist.most_common())}")
    upload_model_file(zpath)
torch.save(TEST_LP, f"{SUB_DIR}/test_logprobs_{SWEEP_NAME}.pt")
upload_model_file(f"{SUB_DIR}/test_logprobs_{SWEEP_NAME}.pt")

refit on all labelled documents: ['rslora', 'full_ft', 'lora_pooled', 'lora_separate']
== refit rslora / all: 90 documents, 10 epochs (the learning-rate decay ends at epoch 10)
training: 836 units/epoch, 105 optimizer steps/epoch, 10 epochs
  epoch 1: train_loss=1.74039, seconds=29.1
  epoch 2: train_loss=1.06278, seconds=58.3
  epoch 3: train_loss=0.66709, seconds=87.2
  epoch 4: train_loss=0.37145, seconds=116.3
  epoch 5: train_loss=0.20665, seconds=145.4
  epoch 6: train_loss=0.11922, seconds=174.4
  epoch 7: train_loss=0.0662, seconds=203.3
  epoch 8: train_loss=0.0403, seconds=232.2
  epoch 9: train_loss=0.02312, seconds=261.2
  epoch 10: train_loss=0.01447, seconds=290.3
   wrote /kaggle/working/p3_submissions_p3s_gqa160/submission_p3s_gqa160_rslora.zip; predicted label counts: {'RatioOfTheDecision': 753, 'Fact': 452, 'PrecedentReliedUpon': 337, 'ArgumentPetitioner': 279, 'Statute': 174, 'RulingByLowerCourt': 102, 'RulingByPresentCourt': 36, 'None': 27, 'ArgumentRespondent': 24,

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded submission_p3s_gqa160_rslora.zip to bucket kavincg23/cs781-models
== refit full_ft / all: 90 documents, 9 epochs (the learning-rate decay ends at epoch 9)
training: 836 units/epoch, 105 optimizer steps/epoch, 9 epochs
  epoch 1: train_loss=1.71404, seconds=35.2
  epoch 2: train_loss=0.99552, seconds=70.4
  epoch 3: train_loss=0.58305, seconds=105.6
  epoch 4: train_loss=0.30203, seconds=140.8
  epoch 5: train_loss=0.16248, seconds=176.0
  epoch 6: train_loss=0.08653, seconds=211.0
  epoch 7: train_loss=0.0463, seconds=246.1
  epoch 8: train_loss=0.02556, seconds=281.0
  epoch 9: train_loss=0.01472, seconds=316.1
   wrote /kaggle/working/p3_submissions_p3s_gqa160/submission_p3s_gqa160_full_ft.zip; predicted label counts: {'RatioOfTheDecision': 737, 'Fact': 444, 'PrecedentReliedUpon': 354, 'ArgumentPetitioner': 265, 'Statute': 175, 'RulingByLowerCourt': 111, 'RulingByPresentCourt': 41, 'ArgumentRespondent': 36, 'None': 26, 'Issue': 8}


Processing Files (0 / 0): |          |  0.00B /  0.00B            

... (11 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded test_logprobs_p3s_gqa160.pt to bucket kavincg23/cs781-models


## 15. Save the summary

In [15]:
SUMMARY = dict(sweep=SWEEP_NAME, model=MODEL_KEY, phase1_checkpoint=PHASE1_CHECKPOINT, phase1_params=PHASE1_PARAMS, sanity_loss=SANITY_LOSS,
               k_folds=K_FOLDS, base=BASE, results=RESULTS, refitted=to_refit, total_minutes=round((time.time() - NB_START) / 60, 1))
path = f"{WORK}/summary_{SWEEP_NAME}.json"
with open(path, "w") as f:
    json.dump(SUMMARY, f, indent=1, default=str)
upload_model_file(path)
# everything the student needs to send back: the zips (to submit), summary_*.json and results_*.json (for the report)
for p in sorted(os.listdir(SUB_DIR)):
    print("  ", f"{SUB_DIR}/{p}", os.path.getsize(f"{SUB_DIR}/{p}") // 1024, "KB")
print(f"done in {(time.time()-NB_START)/60:.0f} min. Send back: summary_{SWEEP_NAME}.json (all numbers) and the zips you want to submit.")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded summary_p3s_gqa160.json to bucket kavincg23/cs781-models
   /kaggle/working/p3_submissions_p3s_gqa160/p3s_gqa160_full_ft_prediction.json 39 KB
   /kaggle/working/p3_submissions_p3s_gqa160/p3s_gqa160_lora_pooled_prediction.json 39 KB
   /kaggle/working/p3_submissions_p3s_gqa160/p3s_gqa160_lora_separate_prediction.json 39 KB
   /kaggle/working/p3_submissions_p3s_gqa160/p3s_gqa160_rslora_prediction.json 39 KB
   /kaggle/working/p3_submissions_p3s_gqa160/submission_p3s_gqa160_full_ft.zip 1 KB
   /kaggle/working/p3_submissions_p3s_gqa160/submission_p3s_gqa160_lora_pooled.zip 1 KB
   /kaggle/working/p3_submissions_p3s_gqa160/submission_p3s_gqa160_lora_separate.zip 1 KB
   /kaggle/working/p3_submissions_p3s_gqa160/submission_p3s_gqa160_rslora.zip 1 KB
   /kaggle/working/p3_submissions_p3s_gqa160/test_logprobs_p3s_gqa160.pt 457 KB
done in 218 min. Send back: summary_p3s_gqa160.json (all numbers) and the zips you want to submit.
